# id1_std_3m 因子


## 裸用回测
该因子为特质波动率因子，含义为个股最近3个月内日收益率序列对中证全指日收益率序列进行一元线性回归的残差的标准差，在华泰的研报当中显示，该因子在市值排名前1/3的股票当中的分层回测效果较好，并且在特定的行业当中表现不佳，所以在策略当中考虑了这两点做了过滤，同时加入了涨跌停的买卖限制和停牌股票的剔除。

In [ ]:
# ======================================================================
# BigQuant / BigTrader 策略：
# id1_std_3m 低特质波动率因子选股
#
# 策略逻辑：
# 1. 构建 id1_std_3m：
#    个股近 3 个月日收益率，对中证全指日收益率做一元线性回归，
#    取回归残差标准差。
# 2. 在总市值排名前 1/3 的股票中选股。
# 3. 剔除指定中信一级行业。
# 4. 选择 id1_std_3m 最低的前 N 只股票。
# 5. 每 K 个交易日等权调仓。
# 6. 剔除停牌股票，买入避开涨停，卖出避开跌停。
# 7. 不使用未来数据：t 日因子只用 t 日及以前数据，BigTrader 日线下单通常在下一根 K 线撮合。
# ======================================================================

import numpy as np
import pandas as pd
from datetime import datetime, timedelta

import dai
from bigmodule import M
from bigtrader.finance.commission import PerOrder


# =========================
# 1. 策略参数
# =========================

START_DATE = "2022-01-01"
END_DATE = "2026-6-20"

CAPITAL_BASE = 1_000_000

# 每次持仓数量：选因子值最低的前 N 只
N_STOCKS = 150

# 每 K 个交易日调仓
K_DAYS = 20

# id1_std_3m：近 3 个月，近似 63 个交易日
REG_WINDOW = 63

# 滚动回归最少有效交易日，避免停牌太多或新股数据不足
MIN_OBS = 50

# 中证全指。BigQuant 不同环境指数后缀可能略有差异，所以代码里做了候选尝试。
MARKET_INDEX_CANDIDATES = ["000985.SH", "000985.SHI", "000985.CSI"]

# 剔除行业：按中信一级行业名称
EXCLUDE_INDUSTRIES = [
    "医药",
    "纺织服装",
    "电力设备",
    "农林牧渔",
    "农林渔牧",   # 兼容用户写法
    "餐饮旅游",
    "石油石化",
    "基础化工",
    "机械",
    "交通运输",
    "传媒",
]

# 手续费：买入万三，卖出千分之一点三，最低 5 元
BUY_COST = 0.0003
SELL_COST = 0.0013
MIN_COST = 5

# 留一点现金缓冲，防止因为涨跌停、手续费、撮合误差导致下单失败
CASH_BUFFER = 0.98


# =========================
# 2. 工具函数
# =========================

def get_query_start_date(start_date: str, buffer_days: int = 260) -> str:
    """
    为了计算滚动 63 个交易日的因子，需要向前多取一段历史数据。
    这里用自然日 260 天作为缓冲。
    """
    dt = datetime.strptime(start_date, "%Y-%m-%d")
    return (dt - timedelta(days=buffer_days)).strftime("%Y-%m-%d")


def normalize_date_col(df: pd.DataFrame, col: str = "date") -> pd.DataFrame:
    df[col] = pd.to_datetime(df[col]).dt.strftime("%Y-%m-%d")
    return df


def query_market_index(start_date: str, end_date: str) -> pd.DataFrame:
    """
    查询中证全指日收盘价。
    如果当前 BigQuant 环境中指数代码后缀不同，会依次尝试几个候选代码。
    """
    last_error = None

    for idx_code in MARKET_INDEX_CANDIDATES:
        try:
            sql = f"""
                SELECT
                    date,
                    instrument,
                    close AS mkt_close
                FROM cn_stock_index_bar1d
                WHERE instrument = '{idx_code}'
                ORDER BY date
            """
            df_idx = dai.query(
                sql,
                filters={"date": [start_date, end_date]}
            ).df()

            if df_idx is not None and len(df_idx) > 0:
                df_idx = normalize_date_col(df_idx)
                df_idx = df_idx.sort_values("date")
                df_idx["mkt_ret"] = df_idx["mkt_close"].pct_change()
                print(f"使用市场指数：{idx_code}")
                return df_idx[["date", "mkt_ret"]]

        except Exception as e:
            last_error = e

    raise ValueError(
        "未能成功读取中证全指行情。请在 BigQuant 中确认中证全指代码，"
        "然后修改 MARKET_INDEX_CANDIDATES。最后一次错误为：{}".format(last_error)
    )


def query_stock_data(start_date: str, end_date: str) -> pd.DataFrame:
    """
    查询股票日行情、状态、市值、行业。
    使用 date filters 是 BigQuant DAI 查询大表时的必要做法。
    """
    sql = """
        SELECT
            b.date,
            b.instrument,
            b.open,
            b.close,
            b.volume,
            b.amount,
            b.upper_limit,
            b.lower_limit,

            fb.total_market_cap,
            fb.list_sector,

            ind.cs_level1_name,

            st.suspended,
            st.price_limit_status,
            st.st_status,
            st.is_risk_warning

        FROM cn_stock_bar1d AS b
        JOIN cn_stock_status AS st
            USING(date, instrument)
        JOIN cn_stock_factors_base AS fb
            USING(date, instrument)
        LEFT JOIN cn_stock_factors_industry AS ind
            USING(date, instrument)

        WHERE
            (
                b.instrument LIKE '%.SH'
                OR b.instrument LIKE '%.SZ'
            )
            AND fb.list_sector IN (1, 2, 3)     -- 主板、创业板、科创板
        ORDER BY b.date, b.instrument
    """

    df = dai.query(
        sql,
        filters={"date": [start_date, end_date]}
    ).df()

    df = normalize_date_col(df)
    df = df.sort_values(["instrument", "date"]).reset_index(drop=True)

    return df


def calc_rolling_capm_resid_std_for_one_stock(
    g: pd.DataFrame,
    window: int = REG_WINDOW,
    min_obs: int = MIN_OBS
) -> pd.Series:
    """
    对单只股票计算滚动 CAPM 残差标准差。

    回归形式：
        r_i,t = alpha_i + beta_i * r_m,t + epsilon_i,t

    id1_std_3m:
        最近 window 个有效交易日 epsilon 的标准差。

    注意：
    - 这里只使用当前日期及以前的数据；
    - 对停牌日、无成交日不参与回归；
    - 使用 SSE / (n - 2) 的回归残差标准误形式。
    """
    g = g.sort_values("date")

    x = g["mkt_ret"].astype(float)
    y = g["ret"].astype(float)

    n = y.rolling(window=window, min_periods=min_obs).count()

    sum_x = x.rolling(window=window, min_periods=min_obs).sum()
    sum_y = y.rolling(window=window, min_periods=min_obs).sum()
    sum_xx = (x * x).rolling(window=window, min_periods=min_obs).sum()
    sum_xy = (x * y).rolling(window=window, min_periods=min_obs).sum()
    sum_yy = (y * y).rolling(window=window, min_periods=min_obs).sum()

    denom = sum_xx - (sum_x * sum_x) / n
    beta = (sum_xy - (sum_x * sum_y) / n) / denom
    alpha = (sum_y / n) - beta * (sum_x / n)

    # SSE = Σ(y - alpha - beta*x)^2
    sse = (
        sum_yy
        + n * alpha * alpha
        + beta * beta * sum_xx
        - 2 * alpha * sum_y
        - 2 * beta * sum_xy
        + 2 * alpha * beta * sum_x
    )

    resid_var = sse / (n - 2)
    resid_var = resid_var.where((n >= min_obs) & (denom > 0) & (resid_var >= 0))

    return np.sqrt(resid_var)


def build_id1_std_3m_signal(
    start_date: str,
    end_date: str,
    n_stocks: int = N_STOCKS
):
    """
    构建每日选股信号：
    date, instrument, id1_std_3m, weight
    """
    query_start = get_query_start_date(start_date)

    print("开始读取股票数据...")
    stock_df = query_stock_data(query_start, end_date)

    print("开始读取中证全指数据...")
    mkt_df = query_market_index(query_start, end_date)

    print("开始计算股票收益率...")
    stock_df["ret"] = (
        stock_df
        .groupby("instrument")["close"]
        .pct_change()
    )

    stock_df = stock_df.merge(mkt_df, on="date", how="left")

    # 只用有效交易日参与滚动回归：剔除停牌、无成交、收益率缺失
    reg_df = stock_df[
        (stock_df["suspended"] == 0)
        & (stock_df["volume"] > 0)
        & (stock_df["amount"] > 0)
        & stock_df["ret"].notna()
        & stock_df["mkt_ret"].notna()
    ].copy()

    reg_df = reg_df.sort_values(["instrument", "date"]).reset_index(drop=True)

    print("开始计算 id1_std_3m，数据量：", len(reg_df))

    reg_df["id1_std_3m"] = (
        reg_df
        .groupby("instrument", group_keys=False)
        .apply(lambda g: calc_rolling_capm_resid_std_for_one_stock(g))
        .reset_index(level=0, drop=True)
    )

    factor_df = reg_df[["date", "instrument", "id1_std_3m"]].copy()

    # 把因子合并回带有市值、行业、交易状态的数据
    all_df = stock_df.merge(
        factor_df,
        on=["date", "instrument"],
        how="left"
    )

    # 只保留正式回测区间
    all_df = all_df[
        (all_df["date"] >= start_date)
        & (all_df["date"] <= end_date)
    ].copy()

    # 保存交易状态，供 handle_data 中判断涨跌停、停牌
    trade_status = all_df[
        [
            "date",
            "instrument",
            "suspended",
            "price_limit_status",
            "volume",
            "amount",
        ]
    ].drop_duplicates(["date", "instrument"]).copy()

    # =========================
    # 选股过滤
    # =========================

    candidate = all_df.copy()

    # 剔除 ST / *ST / 风险警示
    candidate = candidate[
        (candidate["st_status"] == 0)
        & (candidate["is_risk_warning"] == 0)
    ]

    # 剔除交易日停牌、无成交股票
    candidate = candidate[
        (candidate["suspended"] == 0)
        & (candidate["volume"] > 0)
        & (candidate["amount"] > 0)
    ]

    # 因子、市值、行业必须有效
    candidate = candidate[
        candidate["id1_std_3m"].notna()
        & candidate["total_market_cap"].notna()
        & candidate["cs_level1_name"].notna()
    ]

    # 剔除指定行业
    candidate = candidate[
        ~candidate["cs_level1_name"].isin(EXCLUDE_INDUSTRIES)
    ]

    # 为了避免在涨停时买入，选股阶段先剔除当日收盘涨停股票
    # price_limit_status: 1=跌停, 2=非涨跌停, 3=涨停
    candidate = candidate[
        candidate["price_limit_status"] != 3
    ]

    # 市值前 1/3：按每个交易日 total_market_cap 从大到小排名
    candidate["mcap_rank_pct"] = (
        candidate
        .groupby("date")["total_market_cap"]
        .rank(method="first", ascending=False)
        / candidate.groupby("date")["instrument"].transform("count")
    )

    candidate = candidate[
        candidate["mcap_rank_pct"] <= (1.0 / 3.0)
    ]

    # id1_std_3m 是反向因子：因子值越低越好
    signal_df = (
        candidate
        .sort_values(["date", "id1_std_3m", "total_market_cap"], ascending=[True, True, False])
        .groupby("date", group_keys=False)
        .head(n_stocks)
        .copy()
    )

    # 等权仓位
    signal_df["weight"] = (
        signal_df
        .groupby("date")["instrument"]
        .transform(lambda x: CASH_BUFFER / len(x))
    )

    signal_df = signal_df[
        ["date", "instrument", "id1_std_3m", "total_market_cap", "cs_level1_name", "weight"]
    ].sort_values(["date", "id1_std_3m"])

    print("信号构建完成。信号行数：", len(signal_df))
    print("覆盖交易日数量：", signal_df["date"].nunique())
    print("覆盖股票数量：", signal_df["instrument"].nunique())

    return signal_df, trade_status


# =========================
# 3. 构建选股信号
# =========================

signal_df, trade_status_df = build_id1_std_3m_signal(
    start_date=START_DATE,
    end_date=END_DATE,
    n_stocks=N_STOCKS
)

# BigTrader 订阅股票池：用信号中出现过的股票即可。
# 为了让已买股票未来能正常卖出，保留所有历史信号中出现过的 instrument。
INSTRUMENTS = sorted(signal_df["instrument"].unique().tolist())


# =========================
# 4. 回测回调函数
# =========================

def initialize(context):
    """
    初始化函数：只执行一次。
    """
    context.set_commission(
        PerOrder(
            buy_cost=BUY_COST,
            sell_cost=SELL_COST,
            min_cost=MIN_COST
        )
    )

    context.n_stocks = N_STOCKS
    context.k_days = K_DAYS

    # 信号表转字典：date -> DataFrame
    context.signal_by_date = {
        d: df.reset_index(drop=True)
        for d, df in signal_df.groupby("date")
    }

    # 交易状态表转字典：date -> {instrument -> status_dict}
    context.trade_status_by_date = {}
    for d, df in trade_status_df.groupby("date"):
        context.trade_status_by_date[d] = (
            df
            .set_index("instrument")[["suspended", "price_limit_status", "volume", "amount"]]
            .to_dict("index")
        )

    print("initialize 完成。")
    print("订阅股票数量：", len(INSTRUMENTS))
    print("调仓周期 K_DAYS：", context.k_days)
    print("每次目标持股 N_STOCKS：", context.n_stocks)


def _get_instrument_key(x):
    """
    兼容持仓字典 key 可能是字符串，也可能是对象的情况。
    """
    return getattr(x, "symbol", str(x))


def _get_holding_instruments(context):
    """
    获取当前持仓股票代码集合。
    """
    positions = context.get_account_positions()
    holding = set()

    for k, pos in positions.items():
        instrument = _get_instrument_key(k)
        amount = getattr(pos, "amount", 0)

        if amount > 0:
            holding.add(instrument)

    return holding


def _get_status(context, date_str, instrument):
    """
    获取某日某股票交易状态。
    若查不到状态，保守处理为不可交易。
    """
    day_status = context.trade_status_by_date.get(date_str, {})
    return day_status.get(
        instrument,
        {
            "suspended": 1,
            "price_limit_status": np.nan,
            "volume": 0,
            "amount": 0,
        }
    )


def _can_buy(context, date_str, instrument):
    """
    买入限制：
    - 停牌不能买；
    - 无成交不能买；
    - 涨停不能买。
    """
    st = _get_status(context, date_str, instrument)

    if st["suspended"] != 0:
        return False

    if st["volume"] <= 0 or st["amount"] <= 0:
        return False

    # price_limit_status: 1=跌停, 2=非涨跌停, 3=涨停
    if st["price_limit_status"] == 3:
        return False

    return True


def _can_sell(context, date_str, instrument):
    """
    卖出限制：
    - 停牌不能卖；
    - 无成交不能卖；
    - 跌停不能卖。
    """
    st = _get_status(context, date_str, instrument)

    if st["suspended"] != 0:
        return False

    if st["volume"] <= 0 or st["amount"] <= 0:
        return False

    # price_limit_status: 1=跌停, 2=非涨跌停, 3=涨停
    if st["price_limit_status"] == 1:
        return False

    return True


def _is_rebalance_day(context, data):
    """
    判断是否调仓日。
    优先使用 BigTrader 的 rebalance_period；
    如果当前环境没有该对象，则退回到 trading_day_index % K_DAYS。
    """
    try:
        return context.rebalance_period.is_signal_date(data.current_dt.date())
    except Exception:
        return context.trading_day_index % context.k_days == 0


def handle_data(context, data):
    """
    每个交易日运行一次。
    """
    today = pd.Timestamp(data.current_dt).strftime("%Y-%m-%d")

    # 非调仓日不交易
    if not _is_rebalance_day(context, data):
        return

    # 当天没有有效信号，则不交易
    if today not in context.signal_by_date:
        return

    today_signal = context.signal_by_date[today].copy()

    if len(today_signal) == 0:
        return

    target_instruments = set(today_signal["instrument"].tolist())
    holding_instruments = _get_holding_instruments(context)

    # =========================
    # 1）先卖出：不在目标池中的股票
    # =========================
    for instrument in sorted(holding_instruments - target_instruments):
        if _can_sell(context, today, instrument):
            context.order_target_percent(instrument, 0)
        else:
            print(f"{today} 无法卖出 {instrument}：停牌、无成交或跌停。")

    # =========================
    # 2）再买入 / 调整：目标池中的股票
    # =========================
    # 再次过滤：避免调仓日当天涨停、停牌、无成交的股票被买入
    buyable = [
        ins for ins in today_signal["instrument"].tolist()
        if _can_buy(context, today, ins)
    ]

    if len(buyable) == 0:
        print(f"{today} 没有可买入标的。")
        return

    weight = CASH_BUFFER / len(buyable)

    for instrument in buyable:
        context.order_target_percent(instrument, weight)

    print(
        f"{today} 调仓完成：目标 {len(target_instruments)} 只，"
        f"实际可买 {len(buyable)} 只，单票目标权重 {weight:.4f}"
    )


# =========================
# 5. 启动 BigTrader 回测
# =========================

backtest_data = {
    "start_date": START_DATE,
    "end_date": END_DATE,
    "market": "cn_stock",
    "instruments": INSTRUMENTS,
}

m = M.bigtrader.v30(
    data=backtest_data,
    start_date="",
    end_date="",
    initialize=initialize,
    handle_data=handle_data,

    capital_base=CAPITAL_BASE,
    frequency="daily",
    product_type="股票",

    # 调仓周期设置
    rebalance_period_type="交易日",
    rebalance_period_days=str(K_DAYS),
    rebalance_period_roll_forward=True,

    # 标准回测模式
    backtest_engine_mode="标准模式",
    before_start_days=0,

    # 成交量限制：1 表示不额外限制成交比例；如果想更保守，可改成 0.025
    volume_limit=1,

    # 买卖都按开盘价撮合
    order_price_field_buy="open",
    order_price_field_sell="open",

    benchmark="沪深300指数",
    plot_charts=True
)

## 因子收益率、IC及分组回测

由于因子在裸用回测阶段暂未体现出明显的优势，于是考虑首先检验因子的因子收益率、IC和分组回测情况

In [ ]:
import os
import gc
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

import dai


# ============================================================
# 1. 参数区
# ============================================================

START_DATE = "2020-06-20"
END_DATE = "2026-06-20"

START_DATE = pd.to_datetime(START_DATE).strftime("%Y-%m-%d")
END_DATE = pd.to_datetime(END_DATE).strftime("%Y-%m-%d")

# id1_std_3m：近3个月，约63个交易日
REG_WINDOW = 63
MIN_OBS = 50

# 每K个交易日做一次因子检验和分组回测
K_DAYS = 40

# 分10组：G1为因子值最低组，G10为因子值最高组
N_GROUPS = 10

# 股票池：总市值排名前1/3
TOP_MCAP_FRAC = 1 / 3

# 每个调仓日向前取多少自然日的数据，用于覆盖63个有效交易日
LOOKBACK_DAYS = 180

# SQL中每批查询多少只股票，避免IN列表过长
BATCH_SIZE = 500

# 是否沿用前面策略里的行业剔除
USE_INDUSTRY_EXCLUSION = False

EXCLUDE_INDUSTRIES = [
    "医药",
    "纺织服装",
    "电力设备",
    "农林牧渔",
    "农林渔牧",
    "餐饮旅游",
    "石油石化",
    "基础化工",
    "机械",
    "交通运输",
    "传媒",
]

# 中证全指代码候选
MARKET_INDEX_CANDIDATES = ["000985.SH", "000985.SHI", "000985.CSI"]


# ============================================================
# 2. 基础工具函数
# ============================================================

def shift_date(date_str: str, days: int) -> str:
    return (
        pd.to_datetime(date_str) + pd.Timedelta(days=days)
    ).strftime("%Y-%m-%d")


def normalize_date_col(df: pd.DataFrame, col: str = "date") -> pd.DataFrame:
    df[col] = pd.to_datetime(df[col]).dt.strftime("%Y-%m-%d")
    return df


def make_in_clause(instruments):
    instruments = [str(x) for x in instruments if pd.notna(x)]
    return "(" + ",".join([f"'{x}'" for x in instruments]) + ")"


def chunk_list(x, batch_size=BATCH_SIZE):
    for i in range(0, len(x), batch_size):
        yield x[i:i + batch_size]


def safe_concat(dfs):
    dfs = [x for x in dfs if x is not None and len(x) > 0]
    if len(dfs) == 0:
        return pd.DataFrame()
    return pd.concat(dfs, ignore_index=True)


# ============================================================
# 3. 数据查询函数：全部改成单期/小批量查询
# ============================================================

def query_market_index(start_date: str, end_date: str) -> pd.DataFrame:
    last_error = None

    for idx_code in MARKET_INDEX_CANDIDATES:
        try:
            sql = f"""
                SELECT
                    date,
                    instrument,
                    close AS mkt_close
                FROM cn_stock_index_bar1d
                WHERE instrument = '{idx_code}'
                ORDER BY date
            """

            df_idx = dai.query(
                sql,
                filters={"date": [start_date, end_date]}
            ).df()

            if df_idx is not None and len(df_idx) > 0:
                df_idx = normalize_date_col(df_idx)
                df_idx = df_idx.sort_values("date").reset_index(drop=True)
                df_idx["mkt_ret"] = df_idx["mkt_close"].pct_change()
                print(f"市场指数使用：{idx_code}")
                return df_idx[["date", "mkt_ret"]]

        except Exception as e:
            last_error = e

    raise ValueError(f"无法读取中证全指行情，请确认指数代码。最后错误：{last_error}")


def query_asof_universe(asof_date: str) -> pd.DataFrame:
    """
    只查询某一个调仓日的股票池信息。
    """
    sql = """
        SELECT
            b.date,
            b.instrument,
            b.volume,
            b.amount,
            fb.total_market_cap,
            fb.list_sector,
            ind.cs_level1_name,
            st.suspended,
            st.price_limit_status,
            st.st_status,
            st.is_risk_warning
        FROM cn_stock_bar1d AS b
        JOIN cn_stock_status AS st
            USING(date, instrument)
        JOIN cn_stock_factors_base AS fb
            USING(date, instrument)
        LEFT JOIN cn_stock_factors_industry AS ind
            USING(date, instrument)
        WHERE
            (
                b.instrument LIKE '%.SH'
                OR b.instrument LIKE '%.SZ'
            )
            AND fb.list_sector IN (1, 2, 3)
        ORDER BY b.instrument
    """

    df = dai.query(sql, filters={"date": [asof_date, asof_date]}).df()
    df = normalize_date_col(df)
    return df


def query_stock_history_by_instruments(
    instruments,
    start_date: str,
    end_date: str,
    batch_size: int = BATCH_SIZE
) -> pd.DataFrame:
    """
    只查询候选股票最近一段时间的历史行情。
    这一步是内存优化的关键：不再读取全市场多年日线。
    """
    instruments = sorted(list(set(instruments)))
    result = []

    for batch in chunk_list(instruments, batch_size):
        in_clause = make_in_clause(batch)

        sql_with_adj = f"""
            SELECT
                b.date,
                b.instrument,
                b.close,
                b.volume,
                b.amount,
                b.adjust_factor,
                st.suspended
            FROM cn_stock_bar1d AS b
            JOIN cn_stock_status AS st
                USING(date, instrument)
            WHERE b.instrument IN {in_clause}
            ORDER BY b.instrument, b.date
        """

        sql_without_adj = f"""
            SELECT
                b.date,
                b.instrument,
                b.close,
                b.volume,
                b.amount,
                st.suspended
            FROM cn_stock_bar1d AS b
            JOIN cn_stock_status AS st
                USING(date, instrument)
            WHERE b.instrument IN {in_clause}
            ORDER BY b.instrument, b.date
        """

        try:
            df = dai.query(
                sql_with_adj,
                filters={"date": [start_date, end_date]}
            ).df()

            if df is not None and len(df) > 0:
                df["adj_close"] = df["close"] * df["adjust_factor"]

        except Exception:
            df = dai.query(
                sql_without_adj,
                filters={"date": [start_date, end_date]}
            ).df()

            if df is not None and len(df) > 0:
                df["adj_close"] = df["close"]

        if df is not None and len(df) > 0:
            df = normalize_date_col(df)
            result.append(df[[
                "date",
                "instrument",
                "adj_close",
                "volume",
                "amount",
                "suspended",
            ]])

    out = safe_concat(result)

    if len(out) > 0:
        out = out.sort_values(["instrument", "date"]).reset_index(drop=True)

    return out


def query_open_by_instruments(
    instruments,
    date: str,
    prefix: str,
    batch_size: int = BATCH_SIZE
) -> pd.DataFrame:
    """
    查询某一天候选股票的复权开盘价和交易状态。
    prefix = buy 或 sell。
    """
    instruments = sorted(list(set(instruments)))
    result = []

    for batch in chunk_list(instruments, batch_size):
        in_clause = make_in_clause(batch)

        sql_with_adj = f"""
            SELECT
                b.date,
                b.instrument,
                b.open,
                b.volume,
                b.amount,
                b.adjust_factor,
                st.suspended,
                st.price_limit_status
            FROM cn_stock_bar1d AS b
            JOIN cn_stock_status AS st
                USING(date, instrument)
            WHERE b.instrument IN {in_clause}
            ORDER BY b.instrument
        """

        sql_without_adj = f"""
            SELECT
                b.date,
                b.instrument,
                b.open,
                b.volume,
                b.amount,
                st.suspended,
                st.price_limit_status
            FROM cn_stock_bar1d AS b
            JOIN cn_stock_status AS st
                USING(date, instrument)
            WHERE b.instrument IN {in_clause}
            ORDER BY b.instrument
        """

        try:
            df = dai.query(
                sql_with_adj,
                filters={"date": [date, date]}
            ).df()

            if df is not None and len(df) > 0:
                df["adj_open"] = df["open"] * df["adjust_factor"]

        except Exception:
            df = dai.query(
                sql_without_adj,
                filters={"date": [date, date]}
            ).df()

            if df is not None and len(df) > 0:
                df["adj_open"] = df["open"]

        if df is not None and len(df) > 0:
            df = normalize_date_col(df)
            df = df[[
                "instrument",
                "adj_open",
                "volume",
                "amount",
                "suspended",
                "price_limit_status",
            ]].copy()

            df = df.rename(columns={
                "adj_open": f"{prefix}_open",
                "volume": f"{prefix}_volume",
                "amount": f"{prefix}_amount",
                "suspended": f"{prefix}_suspended",
                "price_limit_status": f"{prefix}_price_limit_status",
            })

            result.append(df)

    return safe_concat(result)


# ============================================================
# 4. 单期 id1_std_3m 因子计算
# ============================================================

def calc_id1_std_3m_asof(
    hist_df: pd.DataFrame,
    mkt_df: pd.DataFrame
) -> pd.DataFrame:
    """
    某一个调仓日截面的 id1_std_3m。

    回归：
        r_i,t = alpha_i + beta_i * r_m,t + eps_i,t

    因子：
        最近63个有效交易日 eps_i,t 的标准差。

    注意：
    这里只使用 asof_date 及以前的数据，不使用未来数据。
    """

    if hist_df is None or len(hist_df) == 0:
        return pd.DataFrame(columns=["instrument", "id1_std_3m"])

    df = hist_df.copy()
    df = df.sort_values(["instrument", "date"]).reset_index(drop=True)

    df["ret"] = df.groupby("instrument")["adj_close"].pct_change()
    df = df.merge(mkt_df[["date", "mkt_ret"]], on="date", how="left")

    df = df[
        (df["suspended"] == 0)
        & (df["volume"] > 0)
        & (df["amount"] > 0)
        & df["ret"].notna()
        & df["mkt_ret"].notna()
    ].copy()

    if len(df) == 0:
        return pd.DataFrame(columns=["instrument", "id1_std_3m"])

    df = df.sort_values(["instrument", "date"])

    # 每只股票只取最近 REG_WINDOW 个有效交易日
    df = df.groupby("instrument", group_keys=False).tail(REG_WINDOW)

    df["x"] = df["mkt_ret"].astype(float)
    df["y"] = df["ret"].astype(float)
    df["xx"] = df["x"] * df["x"]
    df["xy"] = df["x"] * df["y"]
    df["yy"] = df["y"] * df["y"]

    agg = df.groupby("instrument").agg(
        n=("y", "count"),
        sum_x=("x", "sum"),
        sum_y=("y", "sum"),
        sum_xx=("xx", "sum"),
        sum_xy=("xy", "sum"),
        sum_yy=("yy", "sum"),
    ).reset_index()

    n = agg["n"].astype(float)
    denom = agg["sum_xx"] - agg["sum_x"] * agg["sum_x"] / n

    beta = (agg["sum_xy"] - agg["sum_x"] * agg["sum_y"] / n) / denom
    alpha = agg["sum_y"] / n - beta * agg["sum_x"] / n

    sse = (
        agg["sum_yy"]
        + n * alpha * alpha
        + beta * beta * agg["sum_xx"]
        - 2 * alpha * agg["sum_y"]
        - 2 * beta * agg["sum_xy"]
        + 2 * alpha * beta * agg["sum_x"]
    )

    resid_var = sse / (n - 2)

    mask = (
        (agg["n"] >= MIN_OBS)
        & (denom > 0)
        & resid_var.notna()
        & (resid_var >= -1e-12)
    )

    out = agg.loc[mask, ["instrument"]].copy()
    out["id1_std_3m"] = np.sqrt(resid_var.loc[mask].clip(lower=0))

    return out


# ============================================================
# 5. 单期截面构建：候选池 + 因子 + 未来收益
# ============================================================

def build_one_period_panel(
    asof_date: str,
    buy_date: str,
    sell_date: str,
    mkt_df: pd.DataFrame
) -> pd.DataFrame:
    """
    构建某个调仓日的一期截面数据：
    date, instrument, id1_std_3m, fwd_ret_k
    """

    # 1）先只查 asof_date 当天股票池和市值，筛出市值前1/3
    uni = query_asof_universe(asof_date)

    if uni is None or len(uni) == 0:
        return pd.DataFrame(columns=["date", "instrument", "id1_std_3m", "fwd_ret_k"])

    uni = uni[
        (uni["suspended"] == 0)
        & (uni["volume"] > 0)
        & (uni["amount"] > 0)
        & (uni["st_status"] == 0)
        & (uni["is_risk_warning"] == 0)
        & (uni["price_limit_status"] != 3)
        & uni["total_market_cap"].notna()
    ].copy()

    if USE_INDUSTRY_EXCLUSION:
        uni = uni[
            uni["cs_level1_name"].notna()
            & (~uni["cs_level1_name"].isin(EXCLUDE_INDUSTRIES))
        ].copy()

    if len(uni) == 0:
        return pd.DataFrame(columns=["date", "instrument", "id1_std_3m", "fwd_ret_k"])

    uni["mcap_rank_pct"] = (
        uni["total_market_cap"].rank(method="first", ascending=False) / len(uni)
    )

    uni = uni[uni["mcap_rank_pct"] <= TOP_MCAP_FRAC].copy()

    instruments = uni["instrument"].dropna().unique().tolist()

    if len(instruments) == 0:
        return pd.DataFrame(columns=["date", "instrument", "id1_std_3m", "fwd_ret_k"])

    # 2）只查询候选股票的近端历史数据来算 id1_std_3m
    window_start = shift_date(asof_date, -LOOKBACK_DAYS)

    hist = query_stock_history_by_instruments(
        instruments=instruments,
        start_date=window_start,
        end_date=asof_date,
    )

    factor = calc_id1_std_3m_asof(hist, mkt_df)

    if len(factor) == 0:
        return pd.DataFrame(columns=["date", "instrument", "id1_std_3m", "fwd_ret_k"])

    # 3）查询买入日和卖出日开盘价
    buy_open = query_open_by_instruments(
        instruments=instruments,
        date=buy_date,
        prefix="buy",
    )

    sell_open = query_open_by_instruments(
        instruments=instruments,
        date=sell_date,
        prefix="sell",
    )

    if len(buy_open) == 0 or len(sell_open) == 0:
        return pd.DataFrame(columns=["date", "instrument", "id1_std_3m", "fwd_ret_k"])

    # 4）合并，计算未来K日收益
    df = (
        uni[["instrument"]]
        .merge(factor, on="instrument", how="inner")
        .merge(buy_open, on="instrument", how="inner")
        .merge(sell_open, on="instrument", how="inner")
    )

    # 买入日：不能停牌、无成交、涨停
    df = df[
        (df["buy_suspended"] == 0)
        & (df["buy_volume"] > 0)
        & (df["buy_amount"] > 0)
        & (df["buy_price_limit_status"] != 3)
    ].copy()

    # 卖出日：不能停牌、无成交、跌停
    df = df[
        (df["sell_suspended"] == 0)
        & (df["sell_volume"] > 0)
        & (df["sell_amount"] > 0)
        & (df["sell_price_limit_status"] != 1)
    ].copy()

    df = df[
        df["buy_open"].notna()
        & df["sell_open"].notna()
        & (df["buy_open"] > 0)
        & (df["sell_open"] > 0)
        & df["id1_std_3m"].notna()
    ].copy()

    if len(df) == 0:
        return pd.DataFrame(columns=["date", "instrument", "id1_std_3m", "fwd_ret_k"])

    df["fwd_ret_k"] = df["sell_open"] / df["buy_open"] - 1
    df["date"] = asof_date

    out = df[["date", "instrument", "id1_std_3m", "fwd_ret_k"]].copy()

    # 主动释放内存
    del uni, hist, factor, buy_open, sell_open, df
    gc.collect()

    return out


# ============================================================
# 6. 因子检验函数
# ============================================================

def calc_cross_section_factor_metrics(g: pd.DataFrame) -> pd.Series:
    """
    每个调仓日做一次截面检验：
    1. 因子收益率：未来K日收益对标准化因子值做截面回归的斜率
    2. IC：Pearson IC
    3. RankIC：Spearman rank IC
    """

    g = g[["id1_std_3m", "fwd_ret_k"]].dropna()

    if len(g) < 30:
        return pd.Series({
            "factor_return_raw": np.nan,
            "ic_raw": np.nan,
            "rank_ic_raw": np.nan,
            "n_stock": len(g),
        })

    x = g["id1_std_3m"].astype(float)
    y = g["fwd_ret_k"].astype(float)

    x_std = x.std(ddof=1)

    if x_std <= 0 or pd.isna(x_std):
        return pd.Series({
            "factor_return_raw": np.nan,
            "ic_raw": np.nan,
            "rank_ic_raw": np.nan,
            "n_stock": len(g),
        })

    x_z = (x - x.mean()) / x_std
    y_dm = y - y.mean()

    factor_return_raw = (x_z * y_dm).sum() / (x_z * x_z).sum()

    ic_raw = x.corr(y)
    rank_ic_raw = x.rank(method="first").corr(y.rank(method="first"))

    return pd.Series({
        "factor_return_raw": factor_return_raw,
        "ic_raw": ic_raw,
        "rank_ic_raw": rank_ic_raw,
        "n_stock": len(g),
    })


def assign_factor_groups(g: pd.DataFrame) -> pd.DataFrame:
    """
    分10组：
    G1 = id1_std_3m最低组，即低特质波动率组
    G10 = id1_std_3m最高组，即高特质波动率组
    """

    g = g.copy()

    if len(g) < N_GROUPS:
        g["group"] = np.nan
        return g

    rank = g["id1_std_3m"].rank(method="first", ascending=True)

    g["group"] = pd.qcut(
        rank,
        q=N_GROUPS,
        labels=[f"G{i}" for i in range(1, N_GROUPS + 1)]
    )

    return g


def calc_perf_metrics(ret: pd.Series, periods_per_year: float) -> dict:
    ret = pd.Series(ret).dropna()

    if len(ret) == 0:
        return {
            "年化收益": np.nan,
            "年化波动": np.nan,
            "夏普": np.nan,
            "最大回撤": np.nan,
            "胜率": np.nan,
            "期末净值": np.nan,
        }

    nav = (1 + ret).cumprod()

    ann_ret = nav.iloc[-1] ** (periods_per_year / len(ret)) - 1
    ann_vol = ret.std(ddof=1) * np.sqrt(periods_per_year)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else np.nan
    max_dd = (nav / nav.cummax() - 1).min()
    win_rate = (ret > 0).mean()

    return {
        "年化收益": ann_ret,
        "年化波动": ann_vol,
        "夏普": sharpe,
        "最大回撤": max_dd,
        "胜率": win_rate,
        "期末净值": nav.iloc[-1],
    }


# ============================================================
# 7. 主流程：逐调仓日计算，不再全量面板计算
# ============================================================

query_start = shift_date(START_DATE, -LOOKBACK_DAYS - 30)

print("读取市场指数数据...")
mkt_df = query_market_index(query_start, END_DATE)

trade_dates = sorted(mkt_df["date"].dropna().unique().tolist())
date_to_idx = {d: i for i, d in enumerate(trade_dates)}

valid_dates = [
    d for d in trade_dates
    if d >= START_DATE
    and d <= END_DATE
    and date_to_idx[d] + K_DAYS + 1 < len(trade_dates)
]

rebalance_dates = valid_dates[::K_DAYS]

print(f"调仓截面数：{len(rebalance_dates)}")
print("开始逐期计算 id1_std_3m；本版不会一次性读取多年全市场数据。")

period_panels = []

for i, asof_date in enumerate(rebalance_dates, 1):
    idx = date_to_idx[asof_date]
    buy_date = trade_dates[idx + 1]
    sell_date = trade_dates[idx + K_DAYS + 1]

    try:
        one = build_one_period_panel(
            asof_date=asof_date,
            buy_date=buy_date,
            sell_date=sell_date,
            mkt_df=mkt_df,
        )

        if len(one) > 0:
            period_panels.append(one)

        if i == 1 or i % 5 == 0 or i == len(rebalance_dates):
            print(
                f"已完成 {i}/{len(rebalance_dates)}："
                f"{asof_date}，样本数 {len(one)}"
            )

    except Exception as e:
        print(f"{asof_date} 计算失败：{e}")

    gc.collect()

if len(period_panels) == 0:
    raise ValueError("没有生成任何有效截面，请检查数据权限、日期区间或股票池过滤条件。")

test_df = pd.concat(period_panels, ignore_index=True)

periods_per_year = 252 / K_DAYS


# ----------------------------
# 7.1 因子收益率、IC、IR
# ----------------------------

factor_ts = (
    test_df
    .groupby("date")
    .apply(calc_cross_section_factor_metrics)
    .dropna()
    .reset_index()
)

raw_factor_ret = factor_ts["factor_return_raw"]
raw_ic = factor_ts["ic_raw"]
raw_rank_ic = factor_ts["rank_ic_raw"]

factor_summary = pd.DataFrame([
    {
        "方向": "原始id1_std_3m",
        "年化因子收益": raw_factor_ret.mean() * periods_per_year,
        "因子收益IR": raw_factor_ret.mean() / raw_factor_ret.std(ddof=1) * np.sqrt(periods_per_year),
        "IC均值": raw_ic.mean(),
        "ICIR": raw_ic.mean() / raw_ic.std(ddof=1) * np.sqrt(periods_per_year),
        "RankIC均值": raw_rank_ic.mean(),
        "RankICIR": raw_rank_ic.mean() / raw_rank_ic.std(ddof=1) * np.sqrt(periods_per_year),
    },
    {
        "方向": "低波动方向(-id1)",
        "年化因子收益": -raw_factor_ret.mean() * periods_per_year,
        "因子收益IR": -raw_factor_ret.mean() / raw_factor_ret.std(ddof=1) * np.sqrt(periods_per_year),
        "IC均值": -raw_ic.mean(),
        "ICIR": -raw_ic.mean() / raw_ic.std(ddof=1) * np.sqrt(periods_per_year),
        "RankIC均值": -raw_rank_ic.mean(),
        "RankICIR": -raw_rank_ic.mean() / raw_rank_ic.std(ddof=1) * np.sqrt(periods_per_year),
    }
])


# ----------------------------
# 7.2 10组分组回测
# ----------------------------

grouped_df = (
    test_df
    .groupby("date", group_keys=False)
    .apply(assign_factor_groups)
    .dropna(subset=["group"])
)

group_ret = (
    grouped_df
    .groupby(["date", "group"])["fwd_ret_k"]
    .mean()
    .unstack()
    .sort_index()
)

group_ret["LS_G1-G10"] = group_ret["G1"] - group_ret["G10"]

group_perf_rows = []

for col in group_ret.columns:
    m = calc_perf_metrics(group_ret[col], periods_per_year)
    m["组合"] = col
    group_perf_rows.append(m)

group_perf = pd.DataFrame(group_perf_rows)
group_perf = group_perf[
    ["组合", "年化收益", "年化波动", "夏普", "最大回撤", "胜率", "期末净值"]
]


# ============================================================
# 8. 美化输出 + 图表
# ============================================================

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from IPython.display import display, Markdown, HTML


pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)


def get_chinese_font():
    candidate_names = [
        "Noto Sans CJK SC",
        "Noto Sans CJK JP",
        "Source Han Sans SC",
        "Source Han Sans CN",
        "WenQuanYi Micro Hei",
        "WenQuanYi Zen Hei",
        "SimHei",
        "Microsoft YaHei",
        "Arial Unicode MS",
        "PingFang SC",
        "Heiti SC",
        "STHeiti",
    ]

    installed_fonts = fm.fontManager.ttflist

    for name in candidate_names:
        for font in installed_fonts:
            if name.lower() in font.name.lower():
                font_prop = fm.FontProperties(fname=font.fname)
                return font_prop, font.name

    candidate_paths = [
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/opentype/noto/NotoSansCJKsc-Regular.otf",
        "/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/truetype/noto/NotoSansSC-Regular.otf",
        "/usr/share/fonts/truetype/wqy/wqy-microhei.ttc",
        "/usr/share/fonts/truetype/wqy/wqy-zenhei.ttc",
        "/usr/share/fonts/opentype/source-han-sans/SourceHanSansSC-Regular.otf",
    ]

    for path in candidate_paths:
        if os.path.exists(path):
            try:
                fm.fontManager.addfont(path)
                font_prop = fm.FontProperties(fname=path)
                return font_prop, font_prop.get_name()
            except Exception:
                pass

    print("警告：未检测到中文字体，图表中文可能显示异常。")
    return fm.FontProperties(), "DejaVu Sans"


CN_FONT, CN_FONT_NAME = get_chinese_font()

mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = [
    CN_FONT_NAME,
    "Noto Sans CJK SC",
    "WenQuanYi Micro Hei",
    "SimHei",
    "Microsoft YaHei",
    "Arial Unicode MS",
    "DejaVu Sans",
]
mpl.rcParams["axes.unicode_minus"] = False
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

print(f"Matplotlib 当前中文字体：{CN_FONT_NAME}")


def set_cn_axis(ax, title=None, xlabel=None, ylabel=None, legend=True):
    if title is not None:
        ax.set_title(title, fontproperties=CN_FONT, fontsize=14)

    if xlabel is not None:
        ax.set_xlabel(xlabel, fontproperties=CN_FONT)

    if ylabel is not None:
        ax.set_ylabel(ylabel, fontproperties=CN_FONT)

    for label in ax.get_xticklabels():
        label.set_fontproperties(CN_FONT)

    for label in ax.get_yticklabels():
        label.set_fontproperties(CN_FONT)

    if legend:
        leg = ax.get_legend()
        if leg is not None:
            for text in leg.get_texts():
                text.set_fontproperties(CN_FONT)


def format_pct(x):
    if pd.isna(x):
        return ""
    return f"{x:.2%}"


def format_num(x):
    if pd.isna(x):
        return ""
    return f"{x:.2f}"


def sort_group_table(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    order = [f"G{i}" for i in range(1, N_GROUPS + 1)] + ["LS_G1-G10"]
    out["组合"] = pd.Categorical(out["组合"], categories=order, ordered=True)
    out = out.sort_values("组合").reset_index(drop=True)
    return out


def make_factor_summary_display(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["年化因子收益"] = out["年化因子收益"].map(format_pct)
    out["因子收益IR"] = out["因子收益IR"].map(format_num)
    out["IC均值"] = out["IC均值"].map(format_pct)
    out["ICIR"] = out["ICIR"].map(format_num)
    out["RankIC均值"] = out["RankIC均值"].map(format_pct)
    out["RankICIR"] = out["RankICIR"].map(format_num)
    return out


def make_group_perf_display(df: pd.DataFrame) -> pd.DataFrame:
    out = sort_group_table(df)
    out["年化收益"] = out["年化收益"].map(format_pct)
    out["年化波动"] = out["年化波动"].map(format_pct)
    out["夏普"] = out["夏普"].map(format_num)
    out["最大回撤"] = out["最大回撤"].map(format_pct)
    out["胜率"] = out["胜率"].map(format_pct)
    out["期末净值"] = out["期末净值"].map(format_num)
    return out


def display_pretty_table(df: pd.DataFrame, title: str):
    display(Markdown(f"### {title}"))

    try:
        styler = df.style.hide(axis="index")
    except Exception:
        styler = df.style.hide_index()

    styler = styler.set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("text-align", "center"),
                    ("font-weight", "bold"),
                    ("background-color", "#f5f5f5"),
                    ("border", "1px solid #d9d9d9"),
                    ("padding", "7px 10px"),
                ],
            },
            {
                "selector": "td",
                "props": [
                    ("text-align", "center"),
                    ("border", "1px solid #e6e6e6"),
                    ("padding", "7px 10px"),
                ],
            },
            {
                "selector": "table",
                "props": [
                    ("border-collapse", "collapse"),
                    ("font-size", "14px"),
                    ("margin-bottom", "18px"),
                ],
            },
        ]
    )

    display(styler)


# ----------------------------
# 8.1 净值曲线数据
# ----------------------------

group_cols = [f"G{i}" for i in range(1, N_GROUPS + 1) if f"G{i}" in group_ret.columns]

plot_cols = group_cols.copy()

if "LS_G1-G10" in group_ret.columns:
    plot_cols.append("LS_G1-G10")

group_ret_plot = group_ret[plot_cols].copy()
group_ret_plot.index = pd.to_datetime(group_ret_plot.index)

group_nav = (1 + group_ret_plot).cumprod()


# ----------------------------
# 8.2 标题和摘要
# ----------------------------

display(Markdown("## id1_std_3m 因子检验结果"))

summary_html = f"""
<table style="border-collapse: collapse; font-size: 14px; margin-bottom: 12px;">
    <tr>
        <td style="padding: 5px 18px 5px 0;"><b>样本区间</b></td>
        <td style="padding: 5px 0;">{START_DATE} ~ {END_DATE}</td>
    </tr>
    <tr>
        <td style="padding: 5px 18px 5px 0;"><b>股票池</b></td>
        <td style="padding: 5px 0;">每日总市值排名前 {TOP_MCAP_FRAC:.0%}</td>
    </tr>
    <tr>
        <td style="padding: 5px 18px 5px 0;"><b>调仓 / 检验频率</b></td>
        <td style="padding: 5px 0;">每 {K_DAYS} 个交易日</td>
    </tr>
    <tr>
        <td style="padding: 5px 18px 5px 0;"><b>有效截面数</b></td>
        <td style="padding: 5px 0;">{factor_ts["date"].nunique()}</td>
    </tr>
    <tr>
        <td style="padding: 5px 18px 5px 0;"><b>平均每期股票数</b></td>
        <td style="padding: 5px 0;">{factor_ts["n_stock"].mean():.0f}</td>
    </tr>
</table>

<p style="font-size: 13px; color: #555;">
说明：G1 为 id1_std_3m 最低组，G10 为 id1_std_3m 最高组；LS_G1-G10 为低波动组减高波动组。
</p>
"""

display(HTML(summary_html))

display_pretty_table(
    make_factor_summary_display(factor_summary),
    "一、因子收益率 / IC / IR"
)

display_pretty_table(
    make_group_perf_display(group_perf),
    "二、10组分组回测绩效"
)


# ----------------------------
# 8.3 图1：10组净值曲线
# ----------------------------

fig, ax = plt.subplots(figsize=(13, 6))

for col in group_cols:
    ax.plot(group_nav.index, group_nav[col], label=col, linewidth=1.6)

ax.legend(ncol=5, fontsize=9)
ax.grid(True, alpha=0.3)

set_cn_axis(
    ax,
    title="10组分组回测净值曲线：G1低波动，G10高波动",
    xlabel="日期",
    ylabel="累计净值",
    legend=True,
)

plt.tight_layout()
plt.show()


# ----------------------------
# 8.4 图2：G1 - G10 多空净值曲线
# ----------------------------

if "LS_G1-G10" in group_nav.columns:
    fig, ax = plt.subplots(figsize=(13, 5))

    ax.plot(
        group_nav.index,
        group_nav["LS_G1-G10"],
        label="LS_G1-G10",
        linewidth=2.2,
    )

    ax.axhline(1.0, linestyle="--", linewidth=1)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

    set_cn_axis(
        ax,
        title="多空组合净值曲线：G1 - G10",
        xlabel="日期",
        ylabel="累计净值",
        legend=True,
    )

    plt.tight_layout()
    plt.show()


# ----------------------------
# 8.5 图3：10组年化收益对比
# ----------------------------

group_perf_bar = sort_group_table(group_perf)
group_perf_bar = group_perf_bar[
    group_perf_bar["组合"].astype(str).str.startswith("G")
].copy()

fig, ax = plt.subplots(figsize=(11, 5))

ax.bar(
    group_perf_bar["组合"].astype(str),
    group_perf_bar["年化收益"],
)

ax.grid(True, axis="y", alpha=0.3)

set_cn_axis(
    ax,
    title="10组年化收益对比",
    xlabel="分组",
    ylabel="年化收益",
    legend=False,
)

for i, v in enumerate(group_perf_bar["年化收益"]):
    if pd.notna(v):
        ax.text(
            i,
            v,
            f"{v:.1%}",
            ha="center",
            va="bottom" if v >= 0 else "top",
            fontsize=9,
            fontproperties=CN_FONT,
        )

plt.tight_layout()
plt.show()


# ----------------------------
# 8.6 图4：IC序列
# ----------------------------

ic_plot = factor_ts.copy()
ic_plot["date"] = pd.to_datetime(ic_plot["date"])
ic_plot = ic_plot.sort_values("date")

ic_plot["cum_ic_raw"] = ic_plot["ic_raw"].cumsum()
ic_plot["cum_rank_ic_raw"] = ic_plot["rank_ic_raw"].cumsum()

fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(
    ic_plot["date"],
    ic_plot["ic_raw"],
    label="IC",
    linewidth=1.4,
)

ax.axhline(0, linestyle="--", linewidth=1)
ax.legend()
ax.grid(True, alpha=0.3)

set_cn_axis(
    ax,
    title="IC 序列",
    xlabel="日期",
    ylabel="IC",
    legend=True,
)

plt.tight_layout()
plt.show()


# ----------------------------
# 8.7 图5：累计IC / 累计RankIC
# ----------------------------

fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(
    ic_plot["date"],
    ic_plot["cum_ic_raw"],
    label="累计 IC",
    linewidth=1.8,
)

ax.plot(
    ic_plot["date"],
    ic_plot["cum_rank_ic_raw"],
    label="累计 RankIC",
    linewidth=1.8,
)

ax.axhline(0, linestyle="--", linewidth=1)
ax.legend()
ax.grid(True, alpha=0.3)

set_cn_axis(
    ax,
    title="累计 IC / 累计 RankIC",
    xlabel="日期",
    ylabel="累计值",
    legend=True,
)

plt.tight_layout()
plt.show()

id1_std_3m 在持有期为20时的原始方向长期偏负，说明高特质波动率股票确实有惩罚效应；但 IC 序列波动和分组非单调说明它不适合直接低波动排序选股。下一步应检验不同持有期、不同 offset 和不同市场状态下，高波动组是否稳定跑输。

In [ ]:
import os
import gc
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

import dai


# ============================================================
# 1. 参数区
# ============================================================

START_DATE = "2024-01-01"
END_DATE = "2026-06-20"

START_DATE = pd.to_datetime(START_DATE).strftime("%Y-%m-%d")
END_DATE = pd.to_datetime(END_DATE).strftime("%Y-%m-%d")

# id1_std_3m：近3个月，约63个交易日
REG_WINDOW = 63
MIN_OBS = 50

# 不同持有期
HOLDING_PERIODS = [40, 60]

# 不同持有期对应的 offset
# offset = 从样本内第几个交易日开始，每 H 个交易日取一次截面
OFFSET_MAP = {
    10: [0, 5],
    20: [0, 5, 10, 15],
    40: [0, 10, 20, 30],
    60: [0, 15, 30, 45],
}

# 分10组：G1为因子值最低组，G10为因子值最高组
N_GROUPS = 10

# 股票池：总市值排名前1/3
TOP_MCAP_FRAC = 1 / 3

# 为了计算近63个有效交易日，向前取180个自然日
LOOKBACK_DAYS = 180

# SQL 每批查询股票数量，避免 IN 列表过长
BATCH_SIZE = 500

# 是否沿用行业剔除
USE_INDUSTRY_EXCLUSION = False

EXCLUDE_INDUSTRIES = [
    "医药",
    "纺织服装",
    "电力设备",
    "农林牧渔",
    "农林渔牧",
    "餐饮旅游",
    "石油石化",
    "基础化工",
    "机械",
    "交通运输",
    "传媒",
]

# 中证全指代码候选
MARKET_INDEX_CANDIDATES = ["000985.SH", "000985.SHI", "000985.CSI"]

# 是否使用本地缓存
USE_DISK_CACHE = True

CACHE_TAG = (
    f"id1_std3m_multi_H_"
    f"{START_DATE.replace('-', '')}_{END_DATE.replace('-', '')}_"
    f"reg{REG_WINDOW}_min{MIN_OBS}_lookback{LOOKBACK_DAYS}_"
    f"top{int(TOP_MCAP_FRAC * 100)}_"
    f"ind{int(USE_INDUSTRY_EXCLUSION)}"
)

CACHE_DIR = f"./{CACHE_TAG}"
FACTOR_CACHE_DIR = os.path.join(CACHE_DIR, "factor_base")
PANEL_CACHE_DIR = os.path.join(CACHE_DIR, "period_panel")

os.makedirs(FACTOR_CACHE_DIR, exist_ok=True)
os.makedirs(PANEL_CACHE_DIR, exist_ok=True)


# ============================================================
# 2. 基础工具函数
# ============================================================

def shift_date(date_str: str, days: int) -> str:
    return (
        pd.to_datetime(date_str) + pd.Timedelta(days=days)
    ).strftime("%Y-%m-%d")


def normalize_date_col(df: pd.DataFrame, col: str = "date") -> pd.DataFrame:
    df[col] = pd.to_datetime(df[col]).dt.strftime("%Y-%m-%d")
    return df


def make_in_clause(instruments):
    instruments = [str(x) for x in instruments if pd.notna(x)]
    return "(" + ",".join([f"'{x}'" for x in instruments]) + ")"


def chunk_list(x, batch_size=BATCH_SIZE):
    for i in range(0, len(x), batch_size):
        yield x[i:i + batch_size]


def safe_concat(dfs):
    dfs = [x for x in dfs if x is not None and len(x) > 0]
    if len(dfs) == 0:
        return pd.DataFrame()
    return pd.concat(dfs, ignore_index=True)


def safe_read_csv(path):
    if os.path.exists(path):
        try:
            return pd.read_csv(path)
        except Exception:
            return None
    return None


def safe_to_csv(df, path):
    try:
        df.to_csv(path, index=False)
    except Exception:
        pass


# ============================================================
# 3. 数据查询函数
# ============================================================

def query_market_index(start_date: str, end_date: str) -> pd.DataFrame:
    """
    查询中证全指行情，并计算市场收益率。
    """

    last_error = None

    for idx_code in MARKET_INDEX_CANDIDATES:
        try:
            sql = f"""
                SELECT
                    date,
                    instrument,
                    close AS mkt_close
                FROM cn_stock_index_bar1d
                WHERE instrument = '{idx_code}'
                ORDER BY date
            """

            df_idx = dai.query(
                sql,
                filters={"date": [start_date, end_date]}
            ).df()

            if df_idx is not None and len(df_idx) > 0:
                df_idx = normalize_date_col(df_idx)
                df_idx = df_idx.sort_values("date").reset_index(drop=True)
                df_idx["mkt_ret"] = df_idx["mkt_close"].pct_change()
                print(f"市场指数使用：{idx_code}")
                return df_idx[["date", "mkt_ret"]]

        except Exception as e:
            last_error = e

    raise ValueError(f"无法读取中证全指行情，请确认指数代码。最后错误：{last_error}")


def query_asof_universe(asof_date: str) -> pd.DataFrame:
    """
    查询某一日股票池、市值、行业、交易状态。
    """

    sql = """
        SELECT
            b.date,
            b.instrument,
            b.volume,
            b.amount,
            fb.total_market_cap,
            fb.list_sector,
            ind.cs_level1_name,
            st.suspended,
            st.price_limit_status,
            st.st_status,
            st.is_risk_warning
        FROM cn_stock_bar1d AS b
        JOIN cn_stock_status AS st
            USING(date, instrument)
        JOIN cn_stock_factors_base AS fb
            USING(date, instrument)
        LEFT JOIN cn_stock_factors_industry AS ind
            USING(date, instrument)
        WHERE
            (
                b.instrument LIKE '%.SH'
                OR b.instrument LIKE '%.SZ'
            )
            AND fb.list_sector IN (1, 2, 3)
        ORDER BY b.instrument
    """

    df = dai.query(sql, filters={"date": [asof_date, asof_date]}).df()
    df = normalize_date_col(df)
    return df


def query_stock_history_by_instruments(
    instruments,
    start_date: str,
    end_date: str,
    batch_size: int = BATCH_SIZE
) -> pd.DataFrame:
    """
    查询候选股票近端历史行情。
    只查市值前1/3候选股票，不查全市场多年数据。
    """

    instruments = sorted(list(set(instruments)))
    result = []

    for batch in chunk_list(instruments, batch_size):
        in_clause = make_in_clause(batch)

        sql_with_adj = f"""
            SELECT
                b.date,
                b.instrument,
                b.close,
                b.volume,
                b.amount,
                b.adjust_factor,
                st.suspended
            FROM cn_stock_bar1d AS b
            JOIN cn_stock_status AS st
                USING(date, instrument)
            WHERE b.instrument IN {in_clause}
            ORDER BY b.instrument, b.date
        """

        sql_without_adj = f"""
            SELECT
                b.date,
                b.instrument,
                b.close,
                b.volume,
                b.amount,
                st.suspended
            FROM cn_stock_bar1d AS b
            JOIN cn_stock_status AS st
                USING(date, instrument)
            WHERE b.instrument IN {in_clause}
            ORDER BY b.instrument, b.date
        """

        try:
            df = dai.query(
                sql_with_adj,
                filters={"date": [start_date, end_date]}
            ).df()

            if df is not None and len(df) > 0:
                df["adj_close"] = df["close"] * df["adjust_factor"]

        except Exception:
            df = dai.query(
                sql_without_adj,
                filters={"date": [start_date, end_date]}
            ).df()

            if df is not None and len(df) > 0:
                df["adj_close"] = df["close"]

        if df is not None and len(df) > 0:
            df = normalize_date_col(df)
            result.append(df[[
                "date",
                "instrument",
                "adj_close",
                "volume",
                "amount",
                "suspended",
            ]])

    out = safe_concat(result)

    if len(out) > 0:
        out = out.sort_values(["instrument", "date"]).reset_index(drop=True)

    return out


def query_open_by_instruments(
    instruments,
    date: str,
    prefix: str,
    batch_size: int = BATCH_SIZE
) -> pd.DataFrame:
    """
    查询某一交易日候选股票复权开盘价和交易状态。
    prefix = buy 或 sell。
    """

    instruments = sorted(list(set(instruments)))
    result = []

    for batch in chunk_list(instruments, batch_size):
        in_clause = make_in_clause(batch)

        sql_with_adj = f"""
            SELECT
                b.date,
                b.instrument,
                b.open,
                b.volume,
                b.amount,
                b.adjust_factor,
                st.suspended,
                st.price_limit_status
            FROM cn_stock_bar1d AS b
            JOIN cn_stock_status AS st
                USING(date, instrument)
            WHERE b.instrument IN {in_clause}
            ORDER BY b.instrument
        """

        sql_without_adj = f"""
            SELECT
                b.date,
                b.instrument,
                b.open,
                b.volume,
                b.amount,
                st.suspended,
                st.price_limit_status
            FROM cn_stock_bar1d AS b
            JOIN cn_stock_status AS st
                USING(date, instrument)
            WHERE b.instrument IN {in_clause}
            ORDER BY b.instrument
        """

        try:
            df = dai.query(
                sql_with_adj,
                filters={"date": [date, date]}
            ).df()

            if df is not None and len(df) > 0:
                df["adj_open"] = df["open"] * df["adjust_factor"]

        except Exception:
            df = dai.query(
                sql_without_adj,
                filters={"date": [date, date]}
            ).df()

            if df is not None and len(df) > 0:
                df["adj_open"] = df["open"]

        if df is not None and len(df) > 0:
            df = normalize_date_col(df)
            df = df[[
                "instrument",
                "adj_open",
                "volume",
                "amount",
                "suspended",
                "price_limit_status",
            ]].copy()

            df = df.rename(columns={
                "adj_open": f"{prefix}_open",
                "volume": f"{prefix}_volume",
                "amount": f"{prefix}_amount",
                "suspended": f"{prefix}_suspended",
                "price_limit_status": f"{prefix}_price_limit_status",
            })

            result.append(df)

    return safe_concat(result)


# ============================================================
# 4. id1_std_3m 因子计算
# ============================================================

def calc_id1_std_3m_asof(
    hist_df: pd.DataFrame,
    mkt_df: pd.DataFrame
) -> pd.DataFrame:
    """
    某一个调仓日截面的 id1_std_3m。

    回归：
        r_i,t = alpha_i + beta_i * r_m,t + eps_i,t

    因子：
        最近63个有效交易日 eps_i,t 的标准差。
    """

    if hist_df is None or len(hist_df) == 0:
        return pd.DataFrame(columns=["instrument", "id1_std_3m"])

    df = hist_df.copy()
    df = df.sort_values(["instrument", "date"]).reset_index(drop=True)

    df["ret"] = df.groupby("instrument")["adj_close"].pct_change()
    df = df.merge(mkt_df[["date", "mkt_ret"]], on="date", how="left")

    df = df[
        (df["suspended"] == 0)
        & (df["volume"] > 0)
        & (df["amount"] > 0)
        & df["ret"].notna()
        & df["mkt_ret"].notna()
    ].copy()

    if len(df) == 0:
        return pd.DataFrame(columns=["instrument", "id1_std_3m"])

    df = df.sort_values(["instrument", "date"])

    # 每只股票只取最近 REG_WINDOW 个有效交易日
    df = df.groupby("instrument", group_keys=False).tail(REG_WINDOW)

    df["x"] = df["mkt_ret"].astype(float)
    df["y"] = df["ret"].astype(float)
    df["xx"] = df["x"] * df["x"]
    df["xy"] = df["x"] * df["y"]
    df["yy"] = df["y"] * df["y"]

    agg = df.groupby("instrument").agg(
        n=("y", "count"),
        sum_x=("x", "sum"),
        sum_y=("y", "sum"),
        sum_xx=("xx", "sum"),
        sum_xy=("xy", "sum"),
        sum_yy=("yy", "sum"),
    ).reset_index()

    n = agg["n"].astype(float)
    denom = agg["sum_xx"] - agg["sum_x"] * agg["sum_x"] / n

    beta = (agg["sum_xy"] - agg["sum_x"] * agg["sum_y"] / n) / denom
    alpha = agg["sum_y"] / n - beta * agg["sum_x"] / n

    sse = (
        agg["sum_yy"]
        + n * alpha * alpha
        + beta * beta * agg["sum_xx"]
        - 2 * alpha * agg["sum_y"]
        - 2 * beta * agg["sum_xy"]
        + 2 * alpha * beta * agg["sum_x"]
    )

    resid_var = sse / (n - 2)

    mask = (
        (agg["n"] >= MIN_OBS)
        & (denom > 0)
        & resid_var.notna()
        & (resid_var >= -1e-12)
    )

    out = agg.loc[mask, ["instrument"]].copy()
    out["id1_std_3m"] = np.sqrt(resid_var.loc[mask].clip(lower=0))

    del df, agg
    gc.collect()

    return out


# ============================================================
# 5. 缓存：某个 asof_date 的因子截面
# ============================================================

def get_factor_cache_path(asof_date: str) -> str:
    return os.path.join(
        FACTOR_CACHE_DIR,
        f"id1_factor_base_{asof_date.replace('-', '')}.csv"
    )


def build_asof_factor_base(asof_date: str, mkt_df: pd.DataFrame) -> pd.DataFrame:
    """
    构建某个调仓日的因子截面：
    1. 当日可交易股票
    2. 总市值前1/3
    3. 计算 id1_std_3m
    """

    cache_path = get_factor_cache_path(asof_date)

    if USE_DISK_CACHE:
        cached = safe_read_csv(cache_path)
        if cached is not None and len(cached) > 0:
            return cached

    uni = query_asof_universe(asof_date)

    if uni is None or len(uni) == 0:
        return pd.DataFrame(columns=["date", "instrument", "id1_std_3m"])

    uni = uni[
        (uni["suspended"] == 0)
        & (uni["volume"] > 0)
        & (uni["amount"] > 0)
        & (uni["st_status"] == 0)
        & (uni["is_risk_warning"] == 0)
        & (uni["price_limit_status"] != 3)
        & uni["total_market_cap"].notna()
    ].copy()

    if USE_INDUSTRY_EXCLUSION:
        uni = uni[
            uni["cs_level1_name"].notna()
            & (~uni["cs_level1_name"].isin(EXCLUDE_INDUSTRIES))
        ].copy()

    if len(uni) == 0:
        return pd.DataFrame(columns=["date", "instrument", "id1_std_3m"])

    uni["mcap_rank_pct"] = (
        uni["total_market_cap"].rank(method="first", ascending=False) / len(uni)
    )

    uni = uni[uni["mcap_rank_pct"] <= TOP_MCAP_FRAC].copy()

    instruments = uni["instrument"].dropna().unique().tolist()

    if len(instruments) == 0:
        return pd.DataFrame(columns=["date", "instrument", "id1_std_3m"])

    window_start = shift_date(asof_date, -LOOKBACK_DAYS)

    hist = query_stock_history_by_instruments(
        instruments=instruments,
        start_date=window_start,
        end_date=asof_date,
    )

    factor = calc_id1_std_3m_asof(hist, mkt_df)

    if factor is None or len(factor) == 0:
        return pd.DataFrame(columns=["date", "instrument", "id1_std_3m"])

    out = (
        uni[["instrument"]]
        .merge(factor, on="instrument", how="inner")
        .copy()
    )

    out["date"] = asof_date
    out = out[["date", "instrument", "id1_std_3m"]].copy()

    if USE_DISK_CACHE:
        safe_to_csv(out, cache_path)

    del uni, hist, factor
    gc.collect()

    return out


# ============================================================
# 6. 构建单期截面：t日因子，t+1开盘买入，t+H+1开盘卖出
# ============================================================

def get_period_panel_cache_path(asof_date: str, buy_date: str, sell_date: str) -> str:
    return os.path.join(
        PANEL_CACHE_DIR,
        f"period_{asof_date.replace('-', '')}_{buy_date.replace('-', '')}_{sell_date.replace('-', '')}.csv"
    )


def build_one_period_panel(
    asof_date: str,
    buy_date: str,
    sell_date: str,
    mkt_df: pd.DataFrame
) -> pd.DataFrame:
    """
    构建单期截面：
    date, instrument, id1_std_3m, fwd_ret
    """

    cache_path = get_period_panel_cache_path(asof_date, buy_date, sell_date)

    if USE_DISK_CACHE:
        cached = safe_read_csv(cache_path)
        if cached is not None and len(cached) > 0:
            return cached

    base = build_asof_factor_base(asof_date, mkt_df)

    if base is None or len(base) == 0:
        return pd.DataFrame(columns=["date", "instrument", "id1_std_3m", "fwd_ret"])

    instruments = base["instrument"].dropna().unique().tolist()

    if len(instruments) == 0:
        return pd.DataFrame(columns=["date", "instrument", "id1_std_3m", "fwd_ret"])

    buy_open = query_open_by_instruments(
        instruments=instruments,
        date=buy_date,
        prefix="buy",
    )

    sell_open = query_open_by_instruments(
        instruments=instruments,
        date=sell_date,
        prefix="sell",
    )

    if buy_open is None or sell_open is None or len(buy_open) == 0 or len(sell_open) == 0:
        return pd.DataFrame(columns=["date", "instrument", "id1_std_3m", "fwd_ret"])

    df = (
        base
        .merge(buy_open, on="instrument", how="inner")
        .merge(sell_open, on="instrument", how="inner")
    )

    # 买入日：不能停牌、无成交、涨停
    df = df[
        (df["buy_suspended"] == 0)
        & (df["buy_volume"] > 0)
        & (df["buy_amount"] > 0)
        & (df["buy_price_limit_status"] != 3)
    ].copy()

    # 卖出日：不能停牌、无成交、跌停
    df = df[
        (df["sell_suspended"] == 0)
        & (df["sell_volume"] > 0)
        & (df["sell_amount"] > 0)
        & (df["sell_price_limit_status"] != 1)
    ].copy()

    df = df[
        df["buy_open"].notna()
        & df["sell_open"].notna()
        & (df["buy_open"] > 0)
        & (df["sell_open"] > 0)
        & df["id1_std_3m"].notna()
    ].copy()

    if len(df) == 0:
        return pd.DataFrame(columns=["date", "instrument", "id1_std_3m", "fwd_ret"])

    df["fwd_ret"] = df["sell_open"] / df["buy_open"] - 1

    out = df[["date", "instrument", "id1_std_3m", "fwd_ret"]].copy()

    if USE_DISK_CACHE:
        safe_to_csv(out, cache_path)

    del base, buy_open, sell_open, df
    gc.collect()

    return out


# ============================================================
# 7. 因子检验函数
# ============================================================

def calc_cross_section_factor_metrics(g: pd.DataFrame) -> pd.Series:
    """
    每个调仓日做一次截面检验：
    1. 因子收益率：未来H日收益对标准化因子值做截面回归的斜率
    2. IC：Pearson IC
    3. RankIC：Spearman rank IC
    """

    g = g[["id1_std_3m", "fwd_ret"]].dropna()

    if len(g) < 30:
        return pd.Series({
            "factor_return_raw": np.nan,
            "ic_raw": np.nan,
            "rank_ic_raw": np.nan,
            "n_stock": len(g),
        })

    x = g["id1_std_3m"].astype(float)
    y = g["fwd_ret"].astype(float)

    x_std = x.std(ddof=1)

    if x_std <= 0 or pd.isna(x_std):
        return pd.Series({
            "factor_return_raw": np.nan,
            "ic_raw": np.nan,
            "rank_ic_raw": np.nan,
            "n_stock": len(g),
        })

    x_z = (x - x.mean()) / x_std
    y_dm = y - y.mean()

    factor_return_raw = (x_z * y_dm).sum() / (x_z * x_z).sum()

    ic_raw = x.corr(y)
    rank_ic_raw = x.rank(method="first").corr(y.rank(method="first"))

    return pd.Series({
        "factor_return_raw": factor_return_raw,
        "ic_raw": ic_raw,
        "rank_ic_raw": rank_ic_raw,
        "n_stock": len(g),
    })


def assign_factor_groups(g: pd.DataFrame) -> pd.DataFrame:
    """
    分10组：
    G1 = id1_std_3m最低组
    G10 = id1_std_3m最高组
    """

    g = g.copy()

    if len(g) < N_GROUPS:
        g["group"] = np.nan
        return g

    rank = g["id1_std_3m"].rank(method="first", ascending=True)

    g["group"] = pd.qcut(
        rank,
        q=N_GROUPS,
        labels=[f"G{i}" for i in range(1, N_GROUPS + 1)]
    )

    return g


def calc_perf_metrics(ret: pd.Series, periods_per_year: float) -> dict:
    ret = pd.Series(ret).dropna()

    if len(ret) == 0:
        return {
            "年化收益": np.nan,
            "年化波动": np.nan,
            "夏普": np.nan,
            "最大回撤": np.nan,
            "胜率": np.nan,
            "期末净值": np.nan,
        }

    nav = (1 + ret).cumprod()

    ann_ret = nav.iloc[-1] ** (periods_per_year / len(ret)) - 1
    ann_vol = ret.std(ddof=1) * np.sqrt(periods_per_year)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else np.nan
    max_dd = (nav / nav.cummax() - 1).min()
    win_rate = (ret > 0).mean()

    return {
        "年化收益": ann_ret,
        "年化波动": ann_vol,
        "夏普": sharpe,
        "最大回撤": max_dd,
        "胜率": win_rate,
        "期末净值": nav.iloc[-1],
    }


def ir_ratio(x, periods_per_year):
    x = pd.Series(x).dropna()
    if len(x) <= 1:
        return np.nan

    std = x.std(ddof=1)

    if std == 0 or pd.isna(std):
        return np.nan

    return x.mean() / std * np.sqrt(periods_per_year)


# ============================================================
# 8. 单个持有期 + offset 的评估
# ============================================================

def get_rebalance_dates_for_experiment(
    trade_dates,
    holding_period: int,
    offset: int
):
    """
    获取某个 holding_period 和 offset 对应的调仓日。
    """

    date_to_idx = {d: i for i, d in enumerate(trade_dates)}

    valid_dates = []

    for d in trade_dates:
        if d < START_DATE or d > END_DATE:
            continue

        idx = date_to_idx[d]

        if idx + holding_period + 1 >= len(trade_dates):
            continue

        buy_date = trade_dates[idx + 1]
        sell_date = trade_dates[idx + holding_period + 1]

        if buy_date > END_DATE or sell_date > END_DATE:
            continue

        valid_dates.append(d)

    if offset >= holding_period:
        raise ValueError(f"offset 必须小于 holding_period：offset={offset}, H={holding_period}")

    return valid_dates[offset::holding_period]


def evaluate_one_experiment(
    holding_period: int,
    offset: int,
    trade_dates,
    mkt_df: pd.DataFrame
):
    """
    评估某一个 holding_period + offset。
    返回：
    1. summary
    2. group_perf
    3. ic_series
    4. group_ret_long
    """

    rebalance_dates = get_rebalance_dates_for_experiment(
        trade_dates=trade_dates,
        holding_period=holding_period,
        offset=offset,
    )

    date_to_idx = {d: i for i, d in enumerate(trade_dates)}

    panels = []

    for i, asof_date in enumerate(rebalance_dates, 1):
        idx = date_to_idx[asof_date]
        buy_date = trade_dates[idx + 1]
        sell_date = trade_dates[idx + holding_period + 1]

        try:
            one = build_one_period_panel(
                asof_date=asof_date,
                buy_date=buy_date,
                sell_date=sell_date,
                mkt_df=mkt_df,
            )

            if one is not None and len(one) > 0:
                panels.append(one)

            if i == 1 or i % 10 == 0 or i == len(rebalance_dates):
                print(
                    f"H={holding_period}, offset={offset}："
                    f"完成 {i}/{len(rebalance_dates)}，日期 {asof_date}，样本数 {len(one)}"
                )

        except Exception as e:
            print(f"H={holding_period}, offset={offset}, date={asof_date} 计算失败：{e}")

        gc.collect()

    if len(panels) == 0:
        empty_summary = pd.DataFrame([{
            "holding_period": holding_period,
            "offset": offset,
            "n_section": 0,
            "avg_n_stock": np.nan,
            "raw_factor_ret_ann": np.nan,
            "raw_factor_ret_ir": np.nan,
            "raw_ic_mean": np.nan,
            "raw_icir": np.nan,
            "raw_rankic_mean": np.nan,
            "raw_rankicir": np.nan,
            "low_factor_ret_ann": np.nan,
            "low_factor_ret_ir": np.nan,
            "low_ic_mean": np.nan,
            "low_icir": np.nan,
            "low_rankic_mean": np.nan,
            "low_rankicir": np.nan,
            "ls_ann_ret": np.nan,
            "ls_sharpe": np.nan,
            "g1_ann_ret": np.nan,
            "g10_ann_ret": np.nan,
        }])

        return empty_summary, pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    test_df_exp = pd.concat(panels, ignore_index=True)

    periods_per_year = 252 / holding_period

    # ----------------------------
    # IC / RankIC / 因子收益率
    # ----------------------------

    factor_ts = (
        test_df_exp
        .groupby("date")
        .apply(calc_cross_section_factor_metrics)
        .dropna()
        .reset_index()
    )

    factor_ts["holding_period"] = holding_period
    factor_ts["offset"] = offset

    # 原始方向：id1越高越差，通常 IC 为负
    # 低波动方向：-id1，因此 IC 取反
    factor_ts["factor_return_lowvol"] = -factor_ts["factor_return_raw"]
    factor_ts["ic_lowvol"] = -factor_ts["ic_raw"]
    factor_ts["rank_ic_lowvol"] = -factor_ts["rank_ic_raw"]

    # ----------------------------
    # 分组收益
    # ----------------------------

    grouped_df = (
        test_df_exp
        .groupby("date", group_keys=False)
        .apply(assign_factor_groups)
        .dropna(subset=["group"])
    )

    group_ret = (
        grouped_df
        .groupby(["date", "group"])["fwd_ret"]
        .mean()
        .unstack()
        .sort_index()
    )

    if "G1" in group_ret.columns and "G10" in group_ret.columns:
        group_ret["LS_G1-G10"] = group_ret["G1"] - group_ret["G10"]

    group_perf_rows = []

    for col in group_ret.columns:
        m = calc_perf_metrics(group_ret[col], periods_per_year)
        m["holding_period"] = holding_period
        m["offset"] = offset
        m["group"] = col
        group_perf_rows.append(m)

    group_perf = pd.DataFrame(group_perf_rows)

    group_ret_long = (
        group_ret
        .reset_index()
        .melt(
            id_vars="date",
            var_name="group",
            value_name="period_ret"
        )
    )

    group_ret_long["holding_period"] = holding_period
    group_ret_long["offset"] = offset

    # ----------------------------
    # 汇总
    # ----------------------------

    def get_group_metric(g, metric):
        x = group_perf[group_perf["group"] == g]
        if len(x) == 0:
            return np.nan
        return x[metric].iloc[0]

    summary = pd.DataFrame([{
        "holding_period": holding_period,
        "offset": offset,
        "n_section": factor_ts["date"].nunique(),
        "avg_n_stock": factor_ts["n_stock"].mean(),

        "raw_factor_ret_ann": factor_ts["factor_return_raw"].mean() * periods_per_year,
        "raw_factor_ret_ir": ir_ratio(factor_ts["factor_return_raw"], periods_per_year),
        "raw_ic_mean": factor_ts["ic_raw"].mean(),
        "raw_icir": ir_ratio(factor_ts["ic_raw"], periods_per_year),
        "raw_rankic_mean": factor_ts["rank_ic_raw"].mean(),
        "raw_rankicir": ir_ratio(factor_ts["rank_ic_raw"], periods_per_year),

        "low_factor_ret_ann": factor_ts["factor_return_lowvol"].mean() * periods_per_year,
        "low_factor_ret_ir": ir_ratio(factor_ts["factor_return_lowvol"], periods_per_year),
        "low_ic_mean": factor_ts["ic_lowvol"].mean(),
        "low_icir": ir_ratio(factor_ts["ic_lowvol"], periods_per_year),
        "low_rankic_mean": factor_ts["rank_ic_lowvol"].mean(),
        "low_rankicir": ir_ratio(factor_ts["rank_ic_lowvol"], periods_per_year),

        "ls_ann_ret": get_group_metric("LS_G1-G10", "年化收益"),
        "ls_sharpe": get_group_metric("LS_G1-G10", "夏普"),
        "g1_ann_ret": get_group_metric("G1", "年化收益"),
        "g10_ann_ret": get_group_metric("G10", "年化收益"),
    }])

    del test_df_exp, grouped_df
    gc.collect()

    return summary, group_perf, factor_ts, group_ret_long


# ============================================================
# 9. 批量运行所有持有期 + offset
# ============================================================

def run_multi_holding_offset_test():
    """
    批量测试不同持有期和不同 offset。
    """

    query_start = shift_date(START_DATE, -LOOKBACK_DAYS - 30)

    print("读取市场指数数据...")
    mkt_df = query_market_index(query_start, END_DATE)

    trade_dates = sorted(mkt_df["date"].dropna().unique().tolist())

    all_summary = []
    all_group_perf = []
    all_ic_series = []
    all_group_ret = []

    for holding_period in HOLDING_PERIODS:
        offsets = OFFSET_MAP.get(holding_period, [0])

        for offset in offsets:
            print("\n" + "=" * 90)
            print(f"开始测试：holding_period={holding_period}, offset={offset}")
            print("=" * 90)

            summary, group_perf, ic_series, group_ret_long = evaluate_one_experiment(
                holding_period=holding_period,
                offset=offset,
                trade_dates=trade_dates,
                mkt_df=mkt_df,
            )

            all_summary.append(summary)

            if len(group_perf) > 0:
                all_group_perf.append(group_perf)

            if len(ic_series) > 0:
                all_ic_series.append(ic_series)

            if len(group_ret_long) > 0:
                all_group_ret.append(group_ret_long)

            gc.collect()

    summary_all = pd.concat(all_summary, ignore_index=True)

    group_perf_all = (
        pd.concat(all_group_perf, ignore_index=True)
        if len(all_group_perf) > 0 else pd.DataFrame()
    )

    ic_series_all = (
        pd.concat(all_ic_series, ignore_index=True)
        if len(all_ic_series) > 0 else pd.DataFrame()
    )

    group_ret_all = (
        pd.concat(all_group_ret, ignore_index=True)
        if len(all_group_ret) > 0 else pd.DataFrame()
    )

    return summary_all, group_perf_all, ic_series_all, group_ret_all


summary_all, group_perf_all, ic_series_all, group_ret_all = run_multi_holding_offset_test()


# ============================================================
# 10. 输出与画图
# ============================================================

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from IPython.display import display, Markdown, HTML


# ----------------------------
# 10.1 中文字体设置
# ----------------------------

def get_chinese_font():
    candidate_names = [
        "Noto Sans CJK SC",
        "Noto Sans CJK JP",
        "Source Han Sans SC",
        "Source Han Sans CN",
        "WenQuanYi Micro Hei",
        "WenQuanYi Zen Hei",
        "SimHei",
        "Microsoft YaHei",
        "Arial Unicode MS",
        "PingFang SC",
        "Heiti SC",
        "STHeiti",
    ]

    installed_fonts = fm.fontManager.ttflist

    for name in candidate_names:
        for font in installed_fonts:
            if name.lower() in font.name.lower():
                font_prop = fm.FontProperties(fname=font.fname)
                return font_prop, font.name

    candidate_paths = [
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/opentype/noto/NotoSansCJKsc-Regular.otf",
        "/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/truetype/noto/NotoSansSC-Regular.otf",
        "/usr/share/fonts/truetype/wqy/wqy-microhei.ttc",
        "/usr/share/fonts/truetype/wqy/wqy-zenhei.ttc",
        "/usr/share/fonts/opentype/source-han-sans/SourceHanSansSC-Regular.otf",
    ]

    for path in candidate_paths:
        if os.path.exists(path):
            try:
                fm.fontManager.addfont(path)
                font_prop = fm.FontProperties(fname=path)
                return font_prop, font_prop.get_name()
            except Exception:
                pass

    print("警告：未检测到中文字体，图表中文可能显示异常。")
    return fm.FontProperties(), "DejaVu Sans"


CN_FONT, CN_FONT_NAME = get_chinese_font()

mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = [
    CN_FONT_NAME,
    "Noto Sans CJK SC",
    "WenQuanYi Micro Hei",
    "SimHei",
    "Microsoft YaHei",
    "Arial Unicode MS",
    "DejaVu Sans",
]
mpl.rcParams["axes.unicode_minus"] = False
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

print(f"Matplotlib 当前中文字体：{CN_FONT_NAME}")


def set_cn_axis(ax, title=None, xlabel=None, ylabel=None, legend=True):
    if title is not None:
        ax.set_title(title, fontproperties=CN_FONT, fontsize=14)

    if xlabel is not None:
        ax.set_xlabel(xlabel, fontproperties=CN_FONT)

    if ylabel is not None:
        ax.set_ylabel(ylabel, fontproperties=CN_FONT)

    for label in ax.get_xticklabels():
        label.set_fontproperties(CN_FONT)

    for label in ax.get_yticklabels():
        label.set_fontproperties(CN_FONT)

    if legend:
        leg = ax.get_legend()
        if leg is not None:
            for text in leg.get_texts():
                text.set_fontproperties(CN_FONT)


# ----------------------------
# 10.2 格式化输出
# ----------------------------

def fmt_pct(x):
    if pd.isna(x):
        return ""
    return f"{x:.2%}"


def fmt_num(x):
    if pd.isna(x):
        return ""
    return f"{x:.2f}"


def display_pretty_table(df: pd.DataFrame, title: str):
    display(Markdown(f"### {title}"))

    try:
        styler = df.style.hide(axis="index")
    except Exception:
        styler = df.style.hide_index()

    styler = styler.set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("text-align", "center"),
                    ("font-weight", "bold"),
                    ("background-color", "#f5f5f5"),
                    ("border", "1px solid #d9d9d9"),
                    ("padding", "7px 10px"),
                ],
            },
            {
                "selector": "td",
                "props": [
                    ("text-align", "center"),
                    ("border", "1px solid #e6e6e6"),
                    ("padding", "7px 10px"),
                ],
            },
            {
                "selector": "table",
                "props": [
                    ("border-collapse", "collapse"),
                    ("font-size", "14px"),
                    ("margin-bottom", "18px"),
                ],
            },
        ]
    )

    display(styler)


def display_summary_table(summary_all):
    cols = [
        "holding_period",
        "offset",
        "n_section",
        "avg_n_stock",
        "low_factor_ret_ann",
        "low_factor_ret_ir",
        "low_ic_mean",
        "low_icir",
        "low_rankic_mean",
        "low_rankicir",
        "ls_ann_ret",
        "ls_sharpe",
        "g1_ann_ret",
        "g10_ann_ret",
    ]

    out = summary_all[cols].copy()

    out = out.rename(columns={
        "holding_period": "持有期",
        "offset": "offset",
        "n_section": "截面数",
        "avg_n_stock": "平均股票数",
        "low_factor_ret_ann": "低波动年化因子收益",
        "low_factor_ret_ir": "低波动因子收益IR",
        "low_ic_mean": "低波动IC均值",
        "low_icir": "低波动ICIR",
        "low_rankic_mean": "低波动RankIC均值",
        "low_rankicir": "低波动RankICIR",
        "ls_ann_ret": "G1-G10年化",
        "ls_sharpe": "G1-G10夏普",
        "g1_ann_ret": "G1年化",
        "g10_ann_ret": "G10年化",
    })

    pct_cols = [
        "低波动年化因子收益",
        "低波动IC均值",
        "低波动RankIC均值",
        "G1-G10年化",
        "G1年化",
        "G10年化",
    ]

    num_cols = [
        "低波动因子收益IR",
        "低波动ICIR",
        "低波动RankICIR",
        "G1-G10夏普",
    ]

    for c in pct_cols:
        out[c] = out[c].map(fmt_pct)

    for c in num_cols:
        out[c] = out[c].map(fmt_num)

    out["平均股票数"] = out["平均股票数"].map(
        lambda x: "" if pd.isna(x) else f"{x:.0f}"
    )

    display_pretty_table(out, "一、不同持有期 / offset 核心结果汇总")


def display_group_perf_table(group_perf_all, holding_period, offset):
    x = group_perf_all[
        (group_perf_all["holding_period"] == holding_period)
        & (group_perf_all["offset"] == offset)
    ].copy()

    if len(x) == 0:
        print(f"没有找到 holding_period={holding_period}, offset={offset} 的分组绩效。")
        return

    order = [f"G{i}" for i in range(1, N_GROUPS + 1)] + ["LS_G1-G10"]
    x["group"] = pd.Categorical(x["group"], categories=order, ordered=True)
    x = x.sort_values("group")

    out = x[[
        "group",
        "年化收益",
        "年化波动",
        "夏普",
        "最大回撤",
        "胜率",
        "期末净值",
    ]].copy()

    out = out.rename(columns={"group": "组合"})

    for c in ["年化收益", "年化波动", "最大回撤", "胜率"]:
        out[c] = out[c].map(fmt_pct)

    for c in ["夏普", "期末净值"]:
        out[c] = out[c].map(fmt_num)

    display_pretty_table(
        out,
        f"二、10组分组回测绩效：持有期={holding_period}日，offset={offset}"
    )


display(Markdown("## id1_std_3m 多持有期 / 多 offset 检验结果"))
display(Markdown(
    f"""
样本区间：`{START_DATE} ~ {END_DATE}`  
股票池：每日总市值排名前 `{TOP_MCAP_FRAC:.0%}`  
分组：`G1` 为 id1_std_3m 最低组，`G10` 为 id1_std_3m 最高组；`LS_G1-G10` 为低波动组减高波动组。
"""
))

display_summary_table(summary_all)


# ============================================================
# 11. 画图函数
# ============================================================

def plot_group_nav(group_ret_all, holding_period, offset):
    x = group_ret_all[
        (group_ret_all["holding_period"] == holding_period)
        & (group_ret_all["offset"] == offset)
    ].copy()

    if len(x) == 0:
        print(f"没有找到 holding_period={holding_period}, offset={offset} 的分组收益。")
        return

    ret = (
        x.pivot(index="date", columns="group", values="period_ret")
        .sort_index()
    )

    ret.index = pd.to_datetime(ret.index)

    group_cols = [f"G{i}" for i in range(1, N_GROUPS + 1) if f"G{i}" in ret.columns]

    nav = (1 + ret).cumprod()

    fig, ax = plt.subplots(figsize=(13, 6))

    for col in group_cols:
        ax.plot(nav.index, nav[col], label=col, linewidth=1.5)

    ax.legend(ncol=5, fontsize=9)
    ax.grid(True, alpha=0.3)

    set_cn_axis(
        ax,
        title=f"G1~G10分组净值曲线：持有期={holding_period}日，offset={offset}",
        xlabel="日期",
        ylabel="累计净值",
        legend=True,
    )

    plt.tight_layout()
    plt.show()


def plot_ls_nav(group_ret_all, holding_period, offset):
    x = group_ret_all[
        (group_ret_all["holding_period"] == holding_period)
        & (group_ret_all["offset"] == offset)
        & (group_ret_all["group"] == "LS_G1-G10")
    ].copy()

    if len(x) == 0:
        print(f"没有找到 holding_period={holding_period}, offset={offset} 的多空收益。")
        return

    x["date"] = pd.to_datetime(x["date"])
    x = x.sort_values("date")

    nav = (1 + x["period_ret"]).cumprod()

    fig, ax = plt.subplots(figsize=(13, 5))

    ax.plot(x["date"], nav, label="LS_G1-G10", linewidth=2.0)
    ax.axhline(1.0, linestyle="--", linewidth=1)
    ax.legend()
    ax.grid(True, alpha=0.3)

    set_cn_axis(
        ax,
        title=f"G1-G10多空净值曲线：持有期={holding_period}日，offset={offset}",
        xlabel="日期",
        ylabel="累计净值",
        legend=True,
    )

    plt.tight_layout()
    plt.show()


def plot_ic_series(ic_series_all, holding_period, offset, direction="lowvol"):
    x = ic_series_all[
        (ic_series_all["holding_period"] == holding_period)
        & (ic_series_all["offset"] == offset)
    ].copy()

    if len(x) == 0:
        print(f"没有找到 holding_period={holding_period}, offset={offset} 的IC序列。")
        return

    x["date"] = pd.to_datetime(x["date"])
    x = x.sort_values("date")

    if direction == "raw":
        ic_col = "ic_raw"
        rankic_col = "rank_ic_raw"
        title_prefix = "原始 id1_std_3m"
    else:
        ic_col = "ic_lowvol"
        rankic_col = "rank_ic_lowvol"
        title_prefix = "低波动方向 -id1_std_3m"

    fig, ax = plt.subplots(figsize=(13, 5))

    ax.plot(x["date"], x[ic_col], label="IC", linewidth=1.5)
    ax.plot(x["date"], x[rankic_col], label="RankIC", linewidth=1.5)
    ax.axhline(0, linestyle="--", linewidth=1)
    ax.legend()
    ax.grid(True, alpha=0.3)

    set_cn_axis(
        ax,
        title=f"{title_prefix}：IC / RankIC 序列，持有期={holding_period}日，offset={offset}",
        xlabel="日期",
        ylabel="IC / RankIC",
        legend=True,
    )

    plt.tight_layout()
    plt.show()


def plot_cum_ic_series(ic_series_all, holding_period, offset, direction="lowvol"):
    x = ic_series_all[
        (ic_series_all["holding_period"] == holding_period)
        & (ic_series_all["offset"] == offset)
    ].copy()

    if len(x) == 0:
        print(f"没有找到 holding_period={holding_period}, offset={offset} 的IC序列。")
        return

    x["date"] = pd.to_datetime(x["date"])
    x = x.sort_values("date")

    if direction == "raw":
        ic_col = "ic_raw"
        rankic_col = "rank_ic_raw"
        title_prefix = "原始 id1_std_3m"
    else:
        ic_col = "ic_lowvol"
        rankic_col = "rank_ic_lowvol"
        title_prefix = "低波动方向 -id1_std_3m"

    x["cum_ic"] = x[ic_col].cumsum()
    x["cum_rankic"] = x[rankic_col].cumsum()

    fig, ax = plt.subplots(figsize=(13, 5))

    ax.plot(x["date"], x["cum_ic"], label="累计IC", linewidth=1.8)
    ax.plot(x["date"], x["cum_rankic"], label="累计RankIC", linewidth=1.8)
    ax.axhline(0, linestyle="--", linewidth=1)
    ax.legend()
    ax.grid(True, alpha=0.3)

    set_cn_axis(
        ax,
        title=f"{title_prefix}：累计IC / 累计RankIC，持有期={holding_period}日，offset={offset}",
        xlabel="日期",
        ylabel="累计值",
        legend=True,
    )

    plt.tight_layout()
    plt.show()


def plot_metric_by_holding_offset(summary_all, metric):
    metric_name_map = {
        "low_factor_ret_ann": "低波动年化因子收益",
        "low_factor_ret_ir": "低波动因子收益IR",
        "low_ic_mean": "低波动IC均值",
        "low_icir": "低波动ICIR",
        "low_rankic_mean": "低波动RankIC均值",
        "low_rankicir": "低波动RankICIR",
        "ls_ann_ret": "G1-G10年化收益",
        "ls_sharpe": "G1-G10夏普",
        "g1_ann_ret": "G1年化收益",
        "g10_ann_ret": "G10年化收益",
    }

    x = summary_all.copy()

    x["combo"] = (
        "H=" + x["holding_period"].astype(str)
        + ", offset=" + x["offset"].astype(str)
    )

    x = x.sort_values(["holding_period", "offset"])

    fig, ax = plt.subplots(figsize=(13, 5))

    ax.bar(x["combo"], x[metric])

    ax.grid(True, axis="y", alpha=0.3)

    set_cn_axis(
        ax,
        title=f"不同持有期 / offset 下的 {metric_name_map.get(metric, metric)}",
        xlabel="组合",
        ylabel=metric_name_map.get(metric, metric),
        legend=False,
    )

    plt.xticks(rotation=45, ha="right")

    plt.tight_layout()
    plt.show()


# ============================================================
# 12. 默认展示一组结果
# ============================================================

SHOW_HOLDING_PERIOD = 20
SHOW_OFFSET = 0

display_group_perf_table(
    group_perf_all,
    holding_period=SHOW_HOLDING_PERIOD,
    offset=SHOW_OFFSET,
)

plot_group_nav(group_ret_all, SHOW_HOLDING_PERIOD, SHOW_OFFSET)
plot_ls_nav(group_ret_all, SHOW_HOLDING_PERIOD, SHOW_OFFSET)

# 低波动方向的 IC / RankIC
plot_ic_series(ic_series_all, SHOW_HOLDING_PERIOD, SHOW_OFFSET, direction="lowvol")
plot_cum_ic_series(ic_series_all, SHOW_HOLDING_PERIOD, SHOW_OFFSET, direction="lowvol")

# 不同持有期 / offset 的核心指标对比
plot_metric_by_holding_offset(summary_all, "low_rankic_mean")
plot_metric_by_holding_offset(summary_all, "low_rankicir")
plot_metric_by_holding_offset(summary_all, "ls_ann_ret")

从以上输出结果来看，将持有期拉长确实可以在一定程度上改善IC的稳定程度，接下来再尝试一下不止在大市值内的选股，而是分别做大市值、中市值和小市值内的回测，观察是否在不同的市值层面上因子会有较大的表现差异

In [ ]:
# ======================================================================
# BigQuant / BigTrader 策略：
# id1_std_3m 低特质波动率因子选股：大市值 / 中市值 / 小市值分别回测
#
# 策略逻辑：
# 1. 构建 id1_std_3m：
#    个股近 3 个月日收益率，对中证全指日收益率做一元线性回归，
#    取回归残差标准差。
# 2. 在剔除行业后的股票池中，按每日总市值分为三组：
#    - 大市值：总市值排名前 1/3
#    - 中市值：总市值排名中间 1/3
#    - 小市值：总市值排名后 1/3
# 3. 三个市值组内分别选择 id1_std_3m 最低的前 N 只股票。
# 4. 每 K 个交易日等权调仓。
# 5. 剔除停牌股票，买入避开涨停，卖出避开跌停。
# 6. 不使用未来数据：t 日因子只用 t 日及以前数据。
# ======================================================================

import numpy as np
import pandas as pd
from datetime import datetime, timedelta

import dai
from bigmodule import M
from bigtrader.finance.commission import PerOrder


# =========================
# 1. 策略参数
# =========================

START_DATE = "2024-01-01"
END_DATE = "2026-6-20"

START_DATE = pd.to_datetime(START_DATE).strftime("%Y-%m-%d")
END_DATE = pd.to_datetime(END_DATE).strftime("%Y-%m-%d")

CAPITAL_BASE = 1_000_000

# 每次持仓数量：每个市值组各选因子值最低的前 N 只
N_STOCKS = 30

# 每 K 个交易日调仓
K_DAYS = 20

# id1_std_3m：近 3 个月，近似 63 个交易日
REG_WINDOW = 63

# 滚动回归最少有效交易日，避免停牌太多或新股数据不足
MIN_OBS = 50

# 中证全指。BigQuant 不同环境指数后缀可能略有差异，所以代码里做了候选尝试。
MARKET_INDEX_CANDIDATES = ["000985.SH", "000985.SHI", "000985.CSI"]

# 剔除行业：按中信一级行业名称
EXCLUDE_INDUSTRIES = [
    "医药",
    "纺织服装",
    "电力设备",
    "农林牧渔",
    "农林渔牧",
    "餐饮旅游",
    "石油石化",
    "基础化工",
    "机械",
    "交通运输",
    "传媒",
]

# 手续费：买入万三，卖出千分之一点三，最低 5 元
BUY_COST = 0.0003
SELL_COST = 0.0013
MIN_COST = 5

# 留一点现金缓冲，防止因为涨跌停、手续费、撮合误差导致下单失败
CASH_BUFFER = 0.98

# 三个市值组定义
SIZE_BUCKETS = {
    "large": {
        "name": "大市值",
        "lower": 0.0,
        "upper": 1.0 / 3.0,
    },
    "mid": {
        "name": "中市值",
        "lower": 1.0 / 3.0,
        "upper": 2.0 / 3.0,
    },
    "small": {
        "name": "小市值",
        "lower": 2.0 / 3.0,
        "upper": 1.0,
    },
}


# =========================
# 2. 工具函数
# =========================

def get_query_start_date(start_date: str, buffer_days: int = 260) -> str:
    """
    为了计算滚动 63 个交易日的因子，需要向前多取一段历史数据。
    这里用自然日 260 天作为缓冲。
    """
    dt = datetime.strptime(start_date, "%Y-%m-%d")
    return (dt - timedelta(days=buffer_days)).strftime("%Y-%m-%d")


def normalize_date_col(df: pd.DataFrame, col: str = "date") -> pd.DataFrame:
    df[col] = pd.to_datetime(df[col]).dt.strftime("%Y-%m-%d")
    return df


def query_market_index(start_date: str, end_date: str) -> pd.DataFrame:
    """
    查询中证全指日收盘价。
    如果当前 BigQuant 环境中指数代码后缀不同，会依次尝试几个候选代码。
    """
    last_error = None

    for idx_code in MARKET_INDEX_CANDIDATES:
        try:
            sql = f"""
                SELECT
                    date,
                    instrument,
                    close AS mkt_close
                FROM cn_stock_index_bar1d
                WHERE instrument = '{idx_code}'
                ORDER BY date
            """

            df_idx = dai.query(
                sql,
                filters={"date": [start_date, end_date]}
            ).df()

            if df_idx is not None and len(df_idx) > 0:
                df_idx = normalize_date_col(df_idx)
                df_idx = df_idx.sort_values("date")
                df_idx["mkt_ret"] = df_idx["mkt_close"].pct_change()
                print(f"使用市场指数：{idx_code}")
                return df_idx[["date", "mkt_ret"]]

        except Exception as e:
            last_error = e

    raise ValueError(
        "未能成功读取中证全指行情。请在 BigQuant 中确认中证全指代码，"
        "然后修改 MARKET_INDEX_CANDIDATES。最后一次错误为：{}".format(last_error)
    )


def query_stock_data(start_date: str, end_date: str) -> pd.DataFrame:
    """
    查询股票日行情、状态、市值、行业。
    使用 date filters 是 BigQuant DAI 查询大表时的必要做法。
    """
    sql = """
        SELECT
            b.date,
            b.instrument,
            b.open,
            b.close,
            b.volume,
            b.amount,
            b.upper_limit,
            b.lower_limit,

            fb.total_market_cap,
            fb.list_sector,

            ind.cs_level1_name,

            st.suspended,
            st.price_limit_status,
            st.st_status,
            st.is_risk_warning

        FROM cn_stock_bar1d AS b
        JOIN cn_stock_status AS st
            USING(date, instrument)
        JOIN cn_stock_factors_base AS fb
            USING(date, instrument)
        LEFT JOIN cn_stock_factors_industry AS ind
            USING(date, instrument)

        WHERE
            (
                b.instrument LIKE '%.SH'
                OR b.instrument LIKE '%.SZ'
            )
            AND fb.list_sector IN (1, 2, 3)
        ORDER BY b.date, b.instrument
    """

    df = dai.query(
        sql,
        filters={"date": [start_date, end_date]}
    ).df()

    df = normalize_date_col(df)
    df = df.sort_values(["instrument", "date"]).reset_index(drop=True)

    return df


def calc_rolling_capm_resid_std_for_one_stock(
    g: pd.DataFrame,
    window: int = REG_WINDOW,
    min_obs: int = MIN_OBS
) -> pd.Series:
    """
    对单只股票计算滚动 CAPM 残差标准差。

    回归形式：
        r_i,t = alpha_i + beta_i * r_m,t + epsilon_i,t

    id1_std_3m:
        最近 window 个有效交易日 epsilon 的标准差。

    注意：
    - 这里只使用当前日期及以前的数据；
    - 对停牌日、无成交日不参与回归；
    - 使用 SSE / (n - 2) 的回归残差标准误形式。
    """
    g = g.sort_values("date")

    x = g["mkt_ret"].astype(float)
    y = g["ret"].astype(float)

    n = y.rolling(window=window, min_periods=min_obs).count()

    sum_x = x.rolling(window=window, min_periods=min_obs).sum()
    sum_y = y.rolling(window=window, min_periods=min_obs).sum()
    sum_xx = (x * x).rolling(window=window, min_periods=min_obs).sum()
    sum_xy = (x * y).rolling(window=window, min_periods=min_obs).sum()
    sum_yy = (y * y).rolling(window=window, min_periods=min_obs).sum()

    denom = sum_xx - (sum_x * sum_x) / n
    beta = (sum_xy - (sum_x * sum_y) / n) / denom
    alpha = (sum_y / n) - beta * (sum_x / n)

    # SSE = Σ(y - alpha - beta*x)^2
    sse = (
        sum_yy
        + n * alpha * alpha
        + beta * beta * sum_xx
        - 2 * alpha * sum_y
        - 2 * beta * sum_xy
        + 2 * alpha * beta * sum_x
    )

    resid_var = sse / (n - 2)
    resid_var = resid_var.where((n >= min_obs) & (denom > 0) & (resid_var >= 0))

    return np.sqrt(resid_var)


def _select_signal_for_size_bucket(
    candidate: pd.DataFrame,
    bucket_key: str,
    n_stocks: int = N_STOCKS
) -> pd.DataFrame:
    """
    在指定市值分组内，选择 id1_std_3m 最低的前 N 只股票。

    mcap_rank_pct：
    - 越小代表市值越大
    - large: <= 1/3
    - mid:   > 1/3 且 <= 2/3
    - small: > 2/3
    """
    bucket = SIZE_BUCKETS[bucket_key]

    lower = bucket["lower"]
    upper = bucket["upper"]

    if bucket_key == "large":
        bucket_df = candidate[
            candidate["mcap_rank_pct"] <= upper
        ].copy()
    elif bucket_key == "mid":
        bucket_df = candidate[
            (candidate["mcap_rank_pct"] > lower)
            & (candidate["mcap_rank_pct"] <= upper)
        ].copy()
    elif bucket_key == "small":
        bucket_df = candidate[
            candidate["mcap_rank_pct"] > lower
        ].copy()
    else:
        raise ValueError(f"未知市值分组：{bucket_key}")

    if len(bucket_df) == 0:
        return pd.DataFrame(
            columns=[
                "date",
                "instrument",
                "id1_std_3m",
                "total_market_cap",
                "cs_level1_name",
                "size_bucket",
                "weight",
            ]
        )

    signal_df = (
        bucket_df
        .sort_values(
            ["date", "id1_std_3m", "total_market_cap"],
            ascending=[True, True, False]
        )
        .groupby("date", group_keys=False)
        .head(n_stocks)
        .copy()
    )

    signal_df["size_bucket"] = bucket["name"]

    # 等权仓位
    signal_df["weight"] = (
        signal_df
        .groupby("date")["instrument"]
        .transform(lambda x: CASH_BUFFER / len(x))
    )

    signal_df = signal_df[
        [
            "date",
            "instrument",
            "id1_std_3m",
            "total_market_cap",
            "cs_level1_name",
            "size_bucket",
            "weight",
        ]
    ].sort_values(["date", "id1_std_3m"])

    return signal_df


def build_id1_std_3m_signals_by_size(
    start_date: str,
    end_date: str,
    n_stocks: int = N_STOCKS
):
    """
    一次性构建三套选股信号：
    - 大市值
    - 中市值
    - 小市值

    返回：
    - signals_by_size: dict[str, DataFrame]
    - trade_status: DataFrame
    """
    query_start = get_query_start_date(start_date)

    print("开始读取股票数据...")
    stock_df = query_stock_data(query_start, end_date)

    print("开始读取中证全指数据...")
    mkt_df = query_market_index(query_start, end_date)

    print("开始计算股票收益率...")
    stock_df["ret"] = (
        stock_df
        .groupby("instrument")["close"]
        .pct_change()
    )

    stock_df = stock_df.merge(mkt_df, on="date", how="left")

    # 只用有效交易日参与滚动回归：剔除停牌、无成交、收益率缺失
    reg_df = stock_df[
        (stock_df["suspended"] == 0)
        & (stock_df["volume"] > 0)
        & (stock_df["amount"] > 0)
        & stock_df["ret"].notna()
        & stock_df["mkt_ret"].notna()
    ].copy()

    reg_df = reg_df.sort_values(["instrument", "date"]).reset_index(drop=True)

    print("开始计算 id1_std_3m，数据量：", len(reg_df))

    reg_df["id1_std_3m"] = (
        reg_df
        .groupby("instrument", group_keys=False)
        .apply(lambda g: calc_rolling_capm_resid_std_for_one_stock(g))
        .reset_index(level=0, drop=True)
    )

    factor_df = reg_df[["date", "instrument", "id1_std_3m"]].copy()

    # 把因子合并回带有市值、行业、交易状态的数据
    all_df = stock_df.merge(
        factor_df,
        on=["date", "instrument"],
        how="left"
    )

    # 只保留正式回测区间
    all_df = all_df[
        (all_df["date"] >= start_date)
        & (all_df["date"] <= end_date)
    ].copy()

    # 保存交易状态，供 handle_data 中判断涨跌停、停牌
    trade_status = all_df[
        [
            "date",
            "instrument",
            "suspended",
            "price_limit_status",
            "volume",
            "amount",
        ]
    ].drop_duplicates(["date", "instrument"]).copy()

    # =========================
    # 选股过滤
    # =========================

    candidate = all_df.copy()

    # 剔除 ST / *ST / 风险警示
    candidate = candidate[
        (candidate["st_status"] == 0)
        & (candidate["is_risk_warning"] == 0)
    ]

    # 剔除交易日停牌、无成交股票
    candidate = candidate[
        (candidate["suspended"] == 0)
        & (candidate["volume"] > 0)
        & (candidate["amount"] > 0)
    ]

    # 因子、市值、行业必须有效
    candidate = candidate[
        candidate["id1_std_3m"].notna()
        & candidate["total_market_cap"].notna()
        & candidate["cs_level1_name"].notna()
    ]

    # 剔除指定行业：保持原逻辑不变
    candidate = candidate[
        ~candidate["cs_level1_name"].isin(EXCLUDE_INDUSTRIES)
    ]

    # 为了避免在涨停时买入，选股阶段先剔除当日收盘涨停股票
    # price_limit_status: 1=跌停, 2=非涨跌停, 3=涨停
    candidate = candidate[
        candidate["price_limit_status"] != 3
    ]

    # 每个交易日按总市值从大到小排名，切成大 / 中 / 小三组
    candidate["mcap_rank_pct"] = (
        candidate
        .groupby("date")["total_market_cap"]
        .rank(method="first", ascending=False)
        / candidate.groupby("date")["instrument"].transform("count")
    )

    # 三个市值组分别选股
    signals_by_size = {}

    for bucket_key, bucket_info in SIZE_BUCKETS.items():
        signal_df = _select_signal_for_size_bucket(
            candidate=candidate,
            bucket_key=bucket_key,
            n_stocks=n_stocks,
        )

        signals_by_size[bucket_key] = signal_df

        print(
            f"{bucket_info['name']}信号构建完成："
            f"信号行数 {len(signal_df)}，"
            f"覆盖交易日 {signal_df['date'].nunique() if len(signal_df) > 0 else 0}，"
            f"覆盖股票 {signal_df['instrument'].nunique() if len(signal_df) > 0 else 0}"
        )

    return signals_by_size, trade_status


# =========================
# 3. 构建三套选股信号
# =========================

signals_by_size, trade_status_df = build_id1_std_3m_signals_by_size(
    start_date=START_DATE,
    end_date=END_DATE,
    n_stocks=N_STOCKS
)

signal_large_df = signals_by_size["large"]
signal_mid_df = signals_by_size["mid"]
signal_small_df = signals_by_size["small"]

INSTRUMENTS_LARGE = sorted(signal_large_df["instrument"].unique().tolist())
INSTRUMENTS_MID = sorted(signal_mid_df["instrument"].unique().tolist())
INSTRUMENTS_SMALL = sorted(signal_small_df["instrument"].unique().tolist())

if len(INSTRUMENTS_LARGE) == 0:
    raise ValueError("大市值信号为空，请检查过滤条件或日期区间。")

if len(INSTRUMENTS_MID) == 0:
    raise ValueError("中市值信号为空，请检查过滤条件或日期区间。")

if len(INSTRUMENTS_SMALL) == 0:
    raise ValueError("小市值信号为空，请检查过滤条件或日期区间。")


# =========================
# 4. 回测回调函数
# =========================

def _prepare_context_common(context, signal_df: pd.DataFrame, bucket_name: str):
    """
    初始化公共逻辑。
    """
    context.set_commission(
        PerOrder(
            buy_cost=BUY_COST,
            sell_cost=SELL_COST,
            min_cost=MIN_COST
        )
    )

    context.n_stocks = N_STOCKS
    context.k_days = K_DAYS
    context.bucket_name = bucket_name

    # 信号表转字典：date -> DataFrame
    context.signal_by_date = {
        d: df.reset_index(drop=True)
        for d, df in signal_df.groupby("date")
    }

    # 交易状态表转字典：date -> {instrument -> status_dict}
    context.trade_status_by_date = {}
    for d, df in trade_status_df.groupby("date"):
        context.trade_status_by_date[d] = (
            df
            .set_index("instrument")[["suspended", "price_limit_status", "volume", "amount"]]
            .to_dict("index")
        )

    print(f"{bucket_name} initialize 完成。")
    print(f"{bucket_name} 订阅股票数量：", signal_df["instrument"].nunique())
    print(f"{bucket_name} 调仓周期 K_DAYS：", context.k_days)
    print(f"{bucket_name} 每次目标持股 N_STOCKS：", context.n_stocks)


def initialize_large(context):
    _prepare_context_common(context, signal_large_df, "大市值")


def initialize_mid(context):
    _prepare_context_common(context, signal_mid_df, "中市值")


def initialize_small(context):
    _prepare_context_common(context, signal_small_df, "小市值")


def _get_instrument_key(x):
    """
    兼容持仓字典 key 可能是字符串，也可能是对象的情况。
    """
    return getattr(x, "symbol", str(x))


def _get_holding_instruments(context):
    """
    获取当前持仓股票代码集合。
    """
    positions = context.get_account_positions()
    holding = set()

    for k, pos in positions.items():
        instrument = _get_instrument_key(k)
        amount = getattr(pos, "amount", 0)

        if amount > 0:
            holding.add(instrument)

    return holding


def _get_status(context, date_str, instrument):
    """
    获取某日某股票交易状态。
    若查不到状态，保守处理为不可交易。
    """
    day_status = context.trade_status_by_date.get(date_str, {})
    return day_status.get(
        instrument,
        {
            "suspended": 1,
            "price_limit_status": np.nan,
            "volume": 0,
            "amount": 0,
        }
    )


def _can_buy(context, date_str, instrument):
    """
    买入限制：
    - 停牌不能买；
    - 无成交不能买；
    - 涨停不能买。
    """
    st = _get_status(context, date_str, instrument)

    if st["suspended"] != 0:
        return False

    if st["volume"] <= 0 or st["amount"] <= 0:
        return False

    # price_limit_status: 1=跌停, 2=非涨跌停, 3=涨停
    if st["price_limit_status"] == 3:
        return False

    return True


def _can_sell(context, date_str, instrument):
    """
    卖出限制：
    - 停牌不能卖；
    - 无成交不能卖；
    - 跌停不能卖。
    """
    st = _get_status(context, date_str, instrument)

    if st["suspended"] != 0:
        return False

    if st["volume"] <= 0 or st["amount"] <= 0:
        return False

    # price_limit_status: 1=跌停, 2=非涨跌停, 3=涨停
    if st["price_limit_status"] == 1:
        return False

    return True


def _is_rebalance_day(context, data):
    """
    判断是否调仓日。
    优先使用 BigTrader 的 rebalance_period；
    如果当前环境没有该对象，则退回到 trading_day_index % K_DAYS。
    """
    try:
        return context.rebalance_period.is_signal_date(data.current_dt.date())
    except Exception:
        return context.trading_day_index % context.k_days == 0


def handle_data(context, data):
    """
    每个交易日运行一次。
    """
    today = pd.Timestamp(data.current_dt).strftime("%Y-%m-%d")

    # 非调仓日不交易
    if not _is_rebalance_day(context, data):
        return

    # 当天没有有效信号，则不交易
    if today not in context.signal_by_date:
        return

    today_signal = context.signal_by_date[today].copy()

    if len(today_signal) == 0:
        return

    target_instruments = set(today_signal["instrument"].tolist())
    holding_instruments = _get_holding_instruments(context)

    # =========================
    # 1）先卖出：不在目标池中的股票
    # =========================
    for instrument in sorted(holding_instruments - target_instruments):
        if _can_sell(context, today, instrument):
            context.order_target_percent(instrument, 0)
        else:
            print(f"{context.bucket_name} {today} 无法卖出 {instrument}：停牌、无成交或跌停。")

    # =========================
    # 2）再买入 / 调整：目标池中的股票
    # =========================
    # 再次过滤：避免调仓日当天涨停、停牌、无成交的股票被买入
    buyable = [
        ins for ins in today_signal["instrument"].tolist()
        if _can_buy(context, today, ins)
    ]

    if len(buyable) == 0:
        print(f"{context.bucket_name} {today} 没有可买入标的。")
        return

    weight = CASH_BUFFER / len(buyable)

    for instrument in buyable:
        context.order_target_percent(instrument, weight)

    print(
        f"{context.bucket_name} {today} 调仓完成："
        f"目标 {len(target_instruments)} 只，"
        f"实际可买 {len(buyable)} 只，"
        f"单票目标权重 {weight:.4f}"
    )


# =========================
# 5. 启动三个 BigTrader 回测
# =========================

def make_backtest_data(instruments):
    return {
        "start_date": START_DATE,
        "end_date": END_DATE,
        "market": "cn_stock",
        "instruments": instruments,
    }


print("\n" + "=" * 80)
print("开始回测：大市值")
print("=" * 80)

m_large = M.bigtrader.v30(
    data=make_backtest_data(INSTRUMENTS_LARGE),
    start_date="",
    end_date="",
    initialize=initialize_large,
    handle_data=handle_data,

    capital_base=CAPITAL_BASE,
    frequency="daily",
    product_type="股票",

    # 调仓周期设置
    rebalance_period_type="交易日",
    rebalance_period_days=str(K_DAYS),
    rebalance_period_roll_forward=True,

    # 标准回测模式
    backtest_engine_mode="标准模式",
    before_start_days=0,

    # 成交量限制：1 表示不额外限制成交比例；如果想更保守，可改成 0.025
    volume_limit=1,

    # 买卖都按开盘价撮合
    order_price_field_buy="open",
    order_price_field_sell="open",

    benchmark="沪深300指数",
    plot_charts=True
)


print("\n" + "=" * 80)
print("开始回测：中市值")
print("=" * 80)

m_mid = M.bigtrader.v30(
    data=make_backtest_data(INSTRUMENTS_MID),
    start_date="",
    end_date="",
    initialize=initialize_mid,
    handle_data=handle_data,

    capital_base=CAPITAL_BASE,
    frequency="daily",
    product_type="股票",

    # 调仓周期设置
    rebalance_period_type="交易日",
    rebalance_period_days=str(K_DAYS),
    rebalance_period_roll_forward=True,

    # 标准回测模式
    backtest_engine_mode="标准模式",
    before_start_days=0,

    volume_limit=1,

    order_price_field_buy="open",
    order_price_field_sell="open",

    benchmark="沪深300指数",
    plot_charts=True
)


print("\n" + "=" * 80)
print("开始回测：小市值")
print("=" * 80)

m_small = M.bigtrader.v30(
    data=make_backtest_data(INSTRUMENTS_SMALL),
    start_date="",
    end_date="",
    initialize=initialize_small,
    handle_data=handle_data,

    capital_base=CAPITAL_BASE,
    frequency="daily",
    product_type="股票",

    # 调仓周期设置
    rebalance_period_type="交易日",
    rebalance_period_days=str(K_DAYS),
    rebalance_period_roll_forward=True,

    # 标准回测模式
    backtest_engine_mode="标准模式",
    before_start_days=0,

    volume_limit=1,

    order_price_field_buy="open",
    order_price_field_sell="open",

    benchmark="沪深300指数",
    plot_charts=True
)


# =========================
# 6. 保存三个回测对象，方便后续查看
# =========================

backtest_results = {
    "大市值": m_large,
    "中市值": m_mid,
    "小市值": m_small,
}

print("\n三个回测已完成，对象已保存到 backtest_results：")
print(backtest_results.keys())

从以上结果来看，该因子确实在大市值层面上表现是最良好的，综合以上的分析，该波动率因子由于其IC不稳定，且单调性不足，不适合作为良好的选股因子，但不难发现，在分组回测时G9-G10等因子值大的分组的表现通常较弱，这可以作为后续过滤股票的一个因子，对该因子的分析为后面其他波动率因子的分析提供了一定的指引。后面又进一步在裸用回测部分调整了持有期以及最大持仓股票数，发现将最大持仓股票数放大可以有效提高该因子的效用，用该因子作为信号做出的策略可以在有些时候控制策略的收益曲线不会过分的随着市场的下跌而下跌，但又能保证在市场行情好的时候跟随市场上涨